# Brain Tumour Segmentation: Region-Adaptive Hybrid CNN-Transformer Ensemble with Internal and External Validation

This notebook trains three complementary 3D segmentation networks on BraTS 2020, fuses them with a **region-adaptive hybrid ensemble**, and tests the result on the held-out BraTS 2020 cases (internal validation) and on BraTS-Africa, an independent cohort from Sub-Saharan African centres (external validation). Every model takes the four MRI sequences of a patient (FLAIR, T1, T1CE, T2) and predicts three nested regions: whole tumour (WT), tumour core (TC) and enhancing tumour (ET).

**Contributions tested here**

1. **Region-adaptive hybrid ensemble.** Two CNNs (nnU-Net, SegResNet) and one transformer (Swin UNETR) are combined with a separate convex weight vector and decision threshold for each tumour region, chosen on the validation cases only. The earlier benchmark showed that the transformers are more sensitive and the CNNs more precise, and that the balance differs between WT, TC and ET; a per-region fusion can use that complementarity where a plain average cannot. Ablations: each member alone, the plain mean ensemble, the CNN-only ensemble, and region-adaptive weights without threshold tuning.
2. **Internal and external validation.** The internal test set is the same 56 patients as the earlier six-network benchmark (paired comparisons stay possible). The external set is the 60 labelled BraTS-Africa cases, never used for training, selection or tuning.
3. **Quality control from architectural disagreement.** The disagreement between the CNN and transformer members is tested as a per-case warning signal (Spearman correlation with Dice and AUROC for detecting failures), internally and externally.
4. **Explainability.** 3D Seg-Grad-CAM maps for every member and region with a quantitative check (share of attribution inside the tumour versus chance), and a sequence-drop ablation that measures how much each MRI sequence contributes to each region.
5. **Protocol-matched comparison with 2D studies.** Besides the standard per-patient 3D metrics, every model is also scored with the slice-wise protocol of 2D studies (Dice over tumour-containing axial slices only), next to the published GCSER-UNet results, so that numbers from the two protocols are not confused.

**Approach.** Native 1 mm resolution (cropped to the brain, no resampling), 128³ training patches with tumour oversampling, nnU-Net-style GPU augmentation (rotation, zoom, mirroring, noise, brightness, contrast, gamma), Dice + BCE loss on the nested regions with deep supervision, Gaussian sliding-window inference with 8-flip test-time augmentation, the BraTS label hierarchy and an ET size filter tuned on validation. Metrics per patient: Dice, HD95 (mm), sensitivity, specificity, with bootstrap 95% CIs and Holm-corrected Wilcoxon tests.

**Data.** BraTS 2020 training cases (`awsaf49/brats20-dataset-training-validation`, 369 patients, train/validation/test 257/56/56 stratified by grade). BraTS-Africa (BraTS 2023 SSA training cases, `aiocta/brats2023-ssa-training-dataset`, 60 patients, ET is label 3).

**Hardware.** Kaggle GPU T4 x2: one training worker per GPU (the transformer on one, the two CNNs on the other), fp16 autocast, micro-batch probing and gradient accumulation, all CPU cores for preprocessing and HD95. Expected wall time about 10 hours; `RESUME_DIR` continues an interrupted run in a new session.

**Kaggle settings.** Accelerator *GPU T4 x2*; Internet *on* (installs MONAI 1.6.1 and einops if missing); add both datasets above with *Add Input*.

**Outputs in `/kaggle/working/`**

```text
src/          volume_lib.py, volume_worker.py, shared_lib.py (written by this notebook)
metrics/      cases.csv, fusion and result tables, statistics, QC, explainability; runs/<run>/ per-run history and per-case metrics
plots/        every figure as PNG
logs/         notebook.log, worker logs, live status and event files
weights/      best-validation checkpoint of every member
predictions/  <run>/<val|test|external>/<case>.npz: TTA probabilities (uint8) on the cropped brain grid
outputs.zip   everything above, created by the last cell
```

The web app's segmentation checkpoint (`weights/deploy/VNet.pth`, the earlier 128³ VNet) is not produced or changed by this notebook.

## 1. Setup

Installs `nibabel`, `einops` and MONAI 1.6.1 only if they are missing (MONAI with `--no-deps`, so Kaggle's PyTorch is never replaced), then imports everything. Expected output: library versions.

In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIREMENTS = {"nibabel": None, "einops": None, "monai": "1.6.1"}  # None: any installed version
for package, version in REQUIREMENTS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed is None or (version is not None and installed != version):
        spec = package if version is None else f"{package}=={version}"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", spec])

import importlib
import json
import os
import shutil
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import monai
import nibabel as nib
import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import FileLink, display
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

print(
    f"torch {torch.__version__} | monai {monai.__version__} | nibabel {nib.__version__} | numpy {np.__version__}",
    flush=True,
)

### Configuration

All settings in one place. Key choices:

- `MODELS`: the ensemble members. Optimiser recipes, deep-supervision weights and the Seg-Grad-CAM layer of each are in `volume_lib.RECIPES`.
- `EPOCHS` per model, each of `ITERS_PER_EPOCH` optimiser steps of `BATCH` patches. The transformer gets fewer epochs because each step costs about three times as much; both GPUs then finish at about the same time.
- `PATCH` is the training patch and sliding window; `SOURCE` is the larger cube read around it so that rotations do not cut corners.
- `FUSION_STEP` and `FUSION_THRESHOLDS` define the search space of the region-adaptive fusion (weights on a 0.1 grid of the simplex, one threshold per region), searched on the validation cases only.
- `VAL_IDS` and `TEST_IDS` fix the validation and test patients of the earlier six-network benchmark, so its results and these are paired by case.
- `REFERENCE_2D` holds the published slice-wise BraTS 2020 Dice of the GCSER-UNet ensemble (Mukherjee et al., arXiv:2605.30510, Table III) for the protocol comparison.
- `QC_FAIL_DICE`: a case with a lower mean Dice counts as a failure in the quality-control analysis.

In [ ]:
DATASET_SLUG = "brats20-dataset-training-validation"  # awsaf49/brats20-dataset-training-validation
EXTERNAL_SLUG = (
    "brats2023-ssa-training-dataset"  # aiocta/brats2023-ssa-training-dataset (BraTS-Africa)
)
MODELS = ["nnU-Net", "SegResNet", "Swin UNETR"]  # two CNNs and one transformer
SEEDS = [42]
PATCH = 128  # training patch and sliding window, voxels at native 1 mm
SOURCE = 160  # cube read around each patch centre before rotation and zoom
EPOCHS = {"nnU-Net": 60, "SegResNet": 60, "Swin UNETR": 30}
ITERS_PER_EPOCH = 250  # optimiser steps per epoch
BATCH = 2  # patches per optimiser step (micro-batch x accumulation)
FG_PROBABILITY = 0.33  # share of patches centred on a tumour voxel
FG_SAMPLES = 4000  # tumour voxel coordinates kept per case and region for that sampling
VAL_EVERY = 5  # epochs between sliding-window validations (and always after the last epoch)
SW_OVERLAP = 0.5
TTA = True  # 8 axis flips at evaluation
ET_CANDIDATES = [0, 50, 100, 200, 300, 500, 800]  # ET size thresholds (voxels) tried on validation
FUSION_STEP = 0.1
FUSION_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
QC_FAIL_DICE = 0.80
CAM_MARGIN = 5  # voxels added around the reference region when scoring Seg-Grad-CAM attribution
SPLIT_SEED = 42  # seeds the bootstrap and the previews
DETERMINISTIC = False
LOADER_WORKERS = max(1, (os.cpu_count() or 4) // 2 - 1)  # per GPU process
LATENCY_ITERS = 4
BOOTSTRAP_SAMPLES = 2000
TRAINING_BUDGET_HOURS = 9.5
RESUME_DIR = None  # e.g. Path("/kaggle/input/<previous-version-output>") to continue a run
REFERENCE_2D = {"model": "GCSER-UNet ensemble (published, 2D)", "WT": 0.95, "TC": 0.92, "ET": 0.90}
# BraTS20_Training_<id> of the earlier benchmark's validation and test cases
VAL_IDS = (
    "003 008 016 017 024 027 038 040 046 081 085 087 102 109 110 116 119 123 129 133 136 156 160 165 178 182 184 195 "
    "197 198 208 210 222 224 227 230 232 239 240 245 252 270 275 283 289 290 300 316 319 327 330 334 344 348 356 360"
)
TEST_IDS = (
    "009 014 021 022 049 051 053 055 059 072 084 088 089 090 092 100 103 107 122 130 131 135 150 152 161 167 170 175 "
    "188 189 190 192 215 236 242 244 250 253 258 266 267 273 276 279 280 292 307 318 326 332 335 347 353 357 364 369"
)

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SRC = WORK / "src"
PLOTS, METRICS, LOGS = WORK / "plots", WORK / "metrics", WORK / "logs"
WEIGHTS, PREDICTIONS = WORK / "weights", WORK / "predictions"
RUNS = METRICS / "runs"
CACHE = Path("/tmp/neuroscan-seg-cache")  # disposable preprocessed volumes, outside the outputs
CASE_CACHE, CLAIMS = CACHE / "cases", CACHE / "claims"
for folder in (SRC, PLOTS, METRICS, LOGS, WEIGHTS, PREDICTIONS, RUNS, CASE_CACHE, CLAIMS):
    folder.mkdir(parents=True, exist_ok=True)

REGIONS = ("WT", "TC", "ET")
ROLES = {"test": "Internal test (BraTS 2020)", "external": "External (BraTS-Africa)"}
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": 0.3})
NOTEBOOK_START = time.time()


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(LOGS / "notebook.log", "a") as handle:
        handle.write(line + "\n")


def save_fig(fig, name):
    path = PLOTS / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    log(f"Saved plot {path.name}")


def save_table(frame, name, index=False, show=True):
    frame.to_csv(METRICS / f"{name}.csv", index=index)
    if show:
        display(frame)


log(
    f"Members: {MODELS} | seeds: {SEEDS} | epochs: {EPOCHS} x {ITERS_PER_EPOCH} steps | patch {PATCH}^3"
)

### Hardware check

GPUs, CPU cores, RAM and free disk in the output folder and the cache, stopping unless two GPUs are visible. Expected output: two Tesla T4 lines.

In [ ]:
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(index)
    log(f"GPU {index}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
memory = psutil.virtual_memory()
log(
    f"CPU cores: {os.cpu_count()} | RAM {memory.total / 1e9:.1f} GB total, {memory.available / 1e9:.1f} GB available"
)
log(
    f"Free disk: /kaggle/working {shutil.disk_usage(WORK).free / 1e9:.1f} GB | cache {shutil.disk_usage(CACHE).free / 1e9:.1f} GB"
)
assert (
    GPU_COUNT >= 2
), "Two GPUs are not visible: select 'GPU T4 x2' as the accelerator in the notebook settings."
DEVICE = torch.device("cuda:0")

### Shared library

Written to `src/volume_lib.py`: case discovery for both naming conventions (BraTS 2020 `_t1ce`, BraTS 2023 `-t1c`), brain cropping and foreground z-score at native resolution, the region bit mask (ET label 4 or 3), deterministic patch sampling with tumour oversampling, GPU augmentation, the three MONAI networks, the deep-supervision Dice + BCE loss, Gaussian sliding-window inference with flip TTA, the region hierarchy and ET filter, per-case Dice, HD95, sensitivity, specificity and slice-wise sums, the fusion search, and Seg-Grad-CAM.

In [ ]:
%%writefile /kaggle/working/src/volume_lib.py
"""Shared code for the BraTS 2020 hybrid CNN-Transformer ensemble segmentation notebook.

The notebook writes this file to /kaggle/working/src so the notebook and the training workers
(one process per GPU) import exactly the same data handling, models, inference, metrics, fusion
and explainability code.
"""

from __future__ import annotations

import itertools
import math
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

MODALITIES = ("flair", "t1", "t1ce", "t2")  # input channel order, as in the web app
# File-name suffixes per sequence: BraTS 2020 (`_t1ce`) and BraTS 2023 (`-t1c`) conventions.
MODALITY_ALIASES = {"flair": ("flair", "t2f"), "t1": ("t1", "t1n"), "t1ce": ("t1ce", "t1c"), "t2": ("t2", "t2w")}
REGIONS = ("WT", "TC", "ET")
HD95_EMPTY = 373.13  # BraTS value when exactly one of prediction and reference is empty

# Family, optimiser recipe, memory settings, deep-supervision weights and the decoder layer used
# for Seg-Grad-CAM (a stage at 1/4 of the input resolution in every network).
RECIPES = {
    "nnU-Net": {
        "family": "CNN",
        "optimizer": "sgd",
        "lr": 1e-2,
        "weight_decay": 3e-5,
        "warmup_epochs": 0,
        "grad_clip": 12.0,
        "micro_batch": 2,
        "ds_weights": (1.0, 0.5, 0.25),
        "cam_layer": "upsamples.2",
    },
    "SegResNet": {
        "family": "CNN",
        "optimizer": "adamw",
        "lr": 2e-4,
        "weight_decay": 1e-5,
        "warmup_epochs": 2,
        "grad_clip": 1.0,
        "micro_batch": 2,
        "ds_weights": (1.0, 0.5, 0.25),
        "cam_layer": "up_layers.0.blocks",
    },
    "Swin UNETR": {
        "family": "Transformer",
        "optimizer": "adamw",
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "warmup_epochs": 2,
        "grad_clip": 1.0,
        "micro_batch": 2,
        "ds_weights": (1.0,),
        "cam_layer": "decoder3",
    },
}


def slug(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def run_id(model, seed):
    return f"{slug(model)}_seed{seed}"


# ---------------------------------------------------------------------------
# Models (MONAI reference implementations; 4 input sequences, 3 sigmoid region outputs)
# ---------------------------------------------------------------------------


def build_model(name):
    """nnU-Net: six-stage residual DynUNet, 32-320 channels, deep supervision at two decoder stages.
    SegResNet: the deep-supervision SegResNet of MONAI Auto3DSeg (Myronenko's BraTS 2018 design).
    Swin UNETR: Swin UNETR-v2, feature size 48, gradient checkpointing."""
    from monai.networks.nets import DynUNet, SegResNetDS, SwinUNETR

    if name == "nnU-Net":
        strides = [1, 2, 2, 2, 2, 2]
        return DynUNet(
            spatial_dims=3,
            in_channels=4,
            out_channels=3,
            kernel_size=[3] * 6,
            strides=strides,
            upsample_kernel_size=strides[1:],
            filters=[32, 64, 128, 256, 320, 320],
            norm_name=("INSTANCE", {"affine": True}),
            deep_supervision=True,
            deep_supr_num=2,
            res_block=True,
        )
    if name == "SegResNet":
        return SegResNetDS(init_filters=32, in_channels=4, out_channels=3, norm="instance", blocks_down=(1, 2, 2, 4), dsdepth=3)
    if name == "Swin UNETR":
        return SwinUNETR(in_channels=4, out_channels=3, feature_size=48, use_checkpoint=True, use_v2=True)
    raise ValueError(f"Unknown model {name!r}")


def output_list(output):
    """Main output first, then deep-supervision outputs, whatever the network returns."""
    if isinstance(output, (list, tuple)):
        return list(output)
    if output.ndim == 6:  # DynUNet stacks its deep-supervision outputs on dim 1 while training
        return list(output.unbind(1))
    return [output]


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------


def case_files(case_dir):
    """Sequence and segmentation files of one case.

    The segmentation is any NIfTI whose name contains "seg", which also finds the misnamed mask of
    BraTS20_Training_355; sequences are recognised by the last token of the file name. A folder
    named like a NIfTI file (Kaggle can unpack an uploaded volume that way) stands for the file in it.
    """
    files = {}
    for path in sorted(Path(case_dir).iterdir()):
        name = path.name.lower()
        if not name.endswith((".nii", ".nii.gz")):
            continue
        if path.is_dir():
            path = next(iter(sorted(path.glob("*.nii*"))), None)
        if path is None or not path.is_file():
            continue
        stem = name.split(".nii")[0]
        if "seg" in stem:
            files["seg"] = path
            continue
        token = re.split(r"[_-]", stem)[-1]
        for modality, aliases in MODALITY_ALIASES.items():
            if token in aliases:
                files[modality] = path
    return files


def zscore_foreground(volume):
    brain = volume != 0
    if brain.any():
        volume[brain] = (volume[brain] - volume[brain].mean()) / (volume[brain].std() + 1e-8)
    return volume


def label_bits(seg, et_label=4):
    """BraTS labels -> bit field: 1 = WT (any tumour), 2 = TC (necrosis and ET), 4 = ET.

    ET is label 4 in BraTS 2020 and label 3 in BraTS 2023 (including BraTS-Africa).
    """
    seg = np.asarray(seg)
    return ((seg > 0) * 1 + np.isin(seg, (1, et_label)) * 2 + (seg == et_label) * 4).astype(np.uint8)


def bits_to_regions(bits):
    return np.stack([(bits & 1) > 0, (bits & 2) > 0, (bits & 4) > 0])


def preprocess_case(args):
    """Load one case, crop it to the bounding box of the brain (any non-zero sequence voxel), z-score
    each sequence over its non-zero voxels and save float16 images and the 3-bit region mask to the
    cache at native 1 mm resolution (run in a worker process)."""
    import nibabel as nib

    torch.set_num_threads(1)
    files, et_label, prefix, fg_samples = args
    volumes = [np.asarray(nib.load(str(files[m])).dataobj, dtype=np.float32) for m in MODALITIES]
    seg_image = nib.load(str(files["seg"]))
    seg = np.rint(np.asarray(seg_image.dataobj, dtype=np.float32)).astype(np.uint8)
    brain = np.any(np.stack(volumes) != 0, axis=0)
    coords = np.argwhere(brain) if brain.any() else np.argwhere(np.ones_like(brain))
    low, high = coords.min(0), coords.max(0) + 1
    box = tuple(slice(int(a), int(b)) for a, b in zip(low, high))
    image = np.stack([zscore_foreground(v[box].copy()) for v in volumes]).astype(np.float16)
    bits = label_bits(seg, et_label)
    crop_bits = np.ascontiguousarray(bits[box])
    np.save(f"{prefix}_img.npy", image)
    np.save(f"{prefix}_lbl.npy", crop_bits)
    spacing = tuple(float(z) for z in seg_image.header.get_zooms()[:3])
    voxel_cm3 = float(np.prod(spacing)) / 1000
    rng = np.random.default_rng(0)
    fg = {}
    for index, region in enumerate(REGIONS):
        points = np.argwhere(bits_to_regions(crop_bits)[index])
        if len(points) > fg_samples:
            points = points[rng.choice(len(points), fg_samples, replace=False)]
        fg[region] = points.astype(np.int16)
    return {
        "shape": seg.shape,
        "spacing": spacing,
        "box_low": [int(v) for v in low],
        "crop_shape": image.shape[1:],
        "volumes_cm3": [float(r.sum()) * voxel_cm3 for r in bits_to_regions(bits)],
        "label_outside_crop": int((bits > 0).sum() - (crop_bits > 0).sum()),
        "labels_present": sorted(int(v) for v in np.unique(seg)),
        "fg": fg,
    }


def load_image(cache_dir, case):
    return np.load(Path(cache_dir) / f"{case}_img.npy", mmap_mode="r")


def load_regions(cache_dir, case):
    return bits_to_regions(np.load(Path(cache_dir) / f"{case}_lbl.npy"))


def crop_pad(array, start, size):
    """Cube of `size` voxels starting at `start` over the last three axes; parts outside are zero."""
    out = np.zeros(array.shape[:-3] + (size,) * 3, dtype=array.dtype)
    src, dst = [], []
    for begin, length in zip(start, array.shape[-3:]):
        a, b = max(int(begin), 0), min(int(begin) + size, length)
        if b <= a:
            return out
        src.append(slice(a, b))
        dst.append(slice(a - int(begin), b - int(begin)))
    out[(..., *dst)] = array[(..., *src)]
    return out


class PatchDataset(torch.utils.data.Dataset):
    """Random training cubes of `source` voxels from the native-resolution cache.

    Item `index` is drawn from a generator seeded by (seed, index), so a run that resumes at an
    epoch boundary sees the same patches as an uninterrupted run. With probability `fg_probability`
    the cube is centred on a random voxel of a random region present in the case (foreground
    oversampling, as in nnU-Net); otherwise the centre is uniform inside the brain box.
    """

    def __init__(self, cache_dir, cases, fg_path, source, patch, fg_probability, seed):
        self.cache_dir, self.cases, self.fg_path = Path(cache_dir), list(cases), fg_path
        self.source, self.patch, self.fg_probability, self.seed = source, patch, fg_probability, seed
        self.fg = None

    def __len__(self):
        return 2**62

    def __getitem__(self, index):
        if self.fg is None:
            with np.load(self.fg_path) as archive:
                self.fg = {key: archive[key] for key in archive.files}
        rng = np.random.default_rng([self.seed, int(index)])
        case = self.cases[rng.integers(len(self.cases))]
        image = load_image(self.cache_dir, case)
        bits = np.load(self.cache_dir / f"{case}_lbl.npy", mmap_mode="r")
        centre = None
        if rng.random() < self.fg_probability:
            present = [key for key in (f"{case}/{r}" for r in REGIONS) if len(self.fg.get(key, ()))]
            if present:
                points = self.fg[present[rng.integers(len(present))]]
                centre = points[rng.integers(len(points))].astype(np.int64)
        if centre is None:
            low = [min(self.patch // 2, s // 2) for s in bits.shape]
            centre = np.array([rng.integers(lo, max(lo, s - self.patch // 2) + 1) for lo, s in zip(low, bits.shape)])
        start = centre - self.source // 2
        return torch.from_numpy(crop_pad(image, start, self.source).astype(np.float32)), torch.from_numpy(crop_pad(bits, start, self.source))


def _rotation(angles):
    """(B, 3) angles in radians -> (B, 3, 3) rotation matrices Rz @ Ry @ Rx."""
    c, s = angles.cos(), angles.sin()
    one, zero = torch.ones_like(c[:, 0]), torch.zeros_like(c[:, 0])

    def matrix(*entries):
        return torch.stack(entries, -1).view(-1, 3, 3)

    rx = matrix(one, zero, zero, zero, c[:, 0], -s[:, 0], zero, s[:, 0], c[:, 0])
    ry = matrix(c[:, 1], zero, s[:, 1], zero, one, zero, -s[:, 1], zero, c[:, 1])
    rz = matrix(c[:, 2], -s[:, 2], zero, s[:, 2], c[:, 2], zero, zero, zero, one)
    return rz @ ry @ rx


def augment(image, bits, patch, max_angle=30.0, zoom_range=(0.7, 1.4), p_rotate=0.2, p_zoom=0.2):
    """GPU augmentation of a batch of source cubes (B, 4, S, S, S) and region bits (B, S, S, S).

    nnU-Net-style recipe: random rotation (up to `max_angle` degrees per axis) and zoom, resampled
    to the central `patch`^3 (bilinear for images, nearest for masks); mirroring on every axis;
    Gaussian noise, brightness, contrast and gamma per sequence. Returns images (B, 4, P, P, P)
    and region targets (B, 3, P, P, P).
    """
    b, device = image.shape[0], image.device
    label = torch.stack([(bits & 1) > 0, (bits & 2) > 0, (bits & 4) > 0], 1).float()
    angles = (torch.rand(b, 3, device=device) * 2 - 1) * math.radians(max_angle)
    angles = angles * (torch.rand(b, 1, device=device) < p_rotate)
    zoom = torch.empty(b, device=device).uniform_(*zoom_range)
    zoom = torch.where(torch.rand(b, device=device) < p_zoom, zoom, torch.ones_like(zoom))
    theta = torch.zeros(b, 3, 4, device=device)
    theta[:, :, :3] = _rotation(angles) / zoom[:, None, None] * (patch / image.shape[-1])
    grid = F.affine_grid(theta, [b, 1, patch, patch, patch], align_corners=False)
    image = F.grid_sample(image, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    label = F.grid_sample(label, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    for axis in (2, 3, 4):
        flip = (torch.rand(b, device=device) < 0.5)[:, None, None, None, None]
        image = torch.where(flip, image.flip(axis), image)
        label = torch.where(flip, label.flip(axis), label)

    sample, channel = (b, 1, 1, 1, 1), (b, image.shape[1], 1, 1, 1)

    def chance(p, shape):
        return torch.rand(shape, device=device) < p

    def uniform(low, high, shape):
        return torch.empty(shape, device=device).uniform_(low, high)

    image = image + torch.randn_like(image) * uniform(0.0, 0.1, sample) * chance(0.15, sample)
    image = image * torch.where(chance(0.15, channel), uniform(0.75, 1.25, channel), 1.0)
    mean = image.mean((2, 3, 4), keepdim=True)
    image = torch.where(chance(0.15, channel), (image - mean) * uniform(0.75, 1.25, channel) + mean, image)
    low = image.amin((2, 3, 4), keepdim=True)
    span = image.amax((2, 3, 4), keepdim=True) - low + 1e-6
    gamma = torch.where(chance(0.15, channel), uniform(0.7, 1.5, channel), 1.0)
    image = ((image - low) / span).clamp(0, 1) ** gamma * span + low
    return image, label


# ---------------------------------------------------------------------------
# Loss, inference, post-processing, metrics
# ---------------------------------------------------------------------------


def region_loss(logits, target):
    """Soft Dice over the batch plus binary cross-entropy, per region."""
    logits = logits.float()
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    dice = 1 - (2 * (prob * target).sum(dims) + 1e-6) / (prob.sum(dims) + target.sum(dims) + 1e-6)
    return dice.mean() + F.binary_cross_entropy_with_logits(logits, target)


def supervised_loss(outputs, target, weights):
    """Deep-supervision loss: weights normalised to sum to one, targets downsampled (nearest) to
    each lower-resolution output."""
    outputs = output_list(outputs)
    total = sum(weights[: len(outputs)])
    loss = 0.0
    for output, weight in zip(outputs, weights):
        scaled = target if output.shape[2:] == target.shape[2:] else F.interpolate(target, size=output.shape[2:], mode="nearest")
        loss = loss + weight / total * region_loss(output, scaled)
    return loss


FLIP_SETS = [(), (2,), (3,), (4,), (2, 3), (2, 4), (3, 4), (2, 3, 4)]


@torch.no_grad()
def sliding_probs(model, image, roi, overlap, tta, sw_batch=2):
    """Sigmoid probabilities of a whole cropped case (1, 4, D, H, W): Gaussian-weighted sliding
    windows of roi^3 voxels, averaged over the 8 axis flips when tta is on.

    Returns the mean (1, 3, D, H, W) and the standard deviation across flips (zero without TTA).
    """
    from monai.inferers import sliding_window_inference

    model.eval()

    def predictor(x):
        with torch.autocast("cuda", dtype=torch.float16, enabled=x.is_cuda):
            return output_list(model(x))[0].float()

    flips = FLIP_SETS if tta else [()]
    total = total_sq = None
    for dims in flips:
        inputs = image.flip(dims) if dims else image
        prob = torch.sigmoid(sliding_window_inference(inputs, (roi,) * 3, sw_batch, predictor, overlap=overlap, mode="gaussian"))
        prob = prob.flip(dims) if dims else prob
        total = prob if total is None else total + prob
        total_sq = prob**2 if total_sq is None else total_sq + prob**2
    mean = total / len(flips)
    std = (total_sq / len(flips) - mean**2).clamp_min(0).sqrt()
    return mean, std


def save_probs(path, probs, tta_std=0.0):
    """Probabilities (3, D, H, W) as compressed uint8 (resolution 1/255) with the case's TTA spread."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, probs=(probs.clamp(0, 1) * 255).round().to(torch.uint8).cpu().numpy(), tta_std=np.float32(tta_std))


def load_probs(path):
    with np.load(path) as archive:
        return archive["probs"], float(archive["tta_std"])


def regions_from_probs(probs, et_min_voxels=0, thresholds=0.5):
    """Nested regions with the BraTS label hierarchy: ET inside TC inside WT.

    `thresholds` is one value or one per region. An ET prediction smaller than et_min_voxels is
    removed, a standard BraTS post-processing step that helps cases without enhancing tumour.
    """
    limit = torch.as_tensor(thresholds, dtype=probs.dtype, device=probs.device).reshape(-1, *[1] * (probs.ndim - 1))
    wt, tc, et = (probs > limit).unbind(0)
    if int(et.sum()) < et_min_voxels:
        et = torch.zeros_like(et)
    tc = tc | et
    wt = wt | tc
    return torch.stack([wt, tc, et])


def dice(pred, ref):
    pred_sum, ref_sum = int(pred.sum()), int(ref.sum())
    if pred_sum + ref_sum == 0:
        return 1.0
    return 2 * int((pred & ref).sum()) / (pred_sum + ref_sum)


def hd95(pred, ref, spacing):
    """95th-percentile symmetric Hausdorff distance in mm (MedPy definition), on the bounding box
    of both masks for speed."""
    from scipy import ndimage

    if not pred.any() and not ref.any():
        return 0.0
    if not pred.any() or not ref.any():
        return HD95_EMPTY
    coords = np.argwhere(pred | ref)
    low = np.maximum(coords.min(0) - 2, 0)
    high = coords.max(0) + 3
    box = tuple(slice(a, b) for a, b in zip(low, high))
    pred = np.pad(pred[box], 2)
    ref = np.pad(ref[box], 2)
    structure = ndimage.generate_binary_structure(3, 1)
    pred_surface = pred ^ ndimage.binary_erosion(pred, structure)
    ref_surface = ref ^ ndimage.binary_erosion(ref, structure)
    to_ref = ndimage.distance_transform_edt(~ref_surface, sampling=spacing)[pred_surface]
    to_pred = ndimage.distance_transform_edt(~pred_surface, sampling=spacing)[ref_surface]
    return float(max(np.percentile(to_ref, 95), np.percentile(to_pred, 95)))


def slice_stats(pred, ref):
    """Sums for the slice-wise protocol of 2D studies, per region: the Dice of every axial slice
    whose reference contains the region (slices without it are skipped, as when only tumour slices
    are kept) and the pooled overlap over those slices."""
    row = {}
    for index, region in enumerate(REGIONS):
        p, r = pred[index], ref[index]
        inter, p_sum, r_sum = (p & r).sum((0, 1)), p.sum((0, 1)), r.sum((0, 1))
        keep = r_sum > 0
        row[f"slice_dice_sum_{region}"] = float((2 * inter[keep] / (p_sum[keep] + r_sum[keep])).sum())
        row[f"slices_{region}"] = int(keep.sum())
        row[f"slice_overlap_{region}"] = int(inter[keep].sum())
        row[f"slice_total_{region}"] = int(p_sum[keep].sum() + r_sum[keep].sum())
    return row


def case_metrics(pred, ref, spacing, total_voxels=None):
    """Dice, HD95, sensitivity, specificity and volumes for the three regions of one case.

    Masks may be cropped to the brain box; `total_voxels` is the native volume size so that
    specificity counts the true negatives outside the box.
    """
    voxel_cm3 = float(np.prod(spacing)) / 1000
    total_voxels = total_voxels or pred[0].size
    row = {}
    for index, region in enumerate(REGIONS):
        p, r = pred[index], ref[index]
        tp = int((p & r).sum())
        fn = int(r.sum()) - tp
        fp = int(p.sum()) - tp
        tn = total_voxels - tp - fn - fp
        row[f"dice_{region}"] = dice(p, r)
        row[f"hd95_{region}"] = hd95(p, r, spacing)
        row[f"sensitivity_{region}"] = tp / (tp + fn) if tp + fn else 1.0
        row[f"specificity_{region}"] = tn / (tn + fp) if tn + fp else 1.0
        row[f"volume_pred_cm3_{region}"] = float(p.sum()) * voxel_cm3
        row[f"volume_ref_cm3_{region}"] = float(r.sum()) * voxel_cm3
    row["dice_mean"] = float(np.mean([row[f"dice_{r}"] for r in REGIONS]))
    row["hd95_mean"] = float(np.mean([row[f"hd95_{r}"] for r in REGIONS]))
    row.update(slice_stats(pred, ref))
    return row


def bootstrap_mean_ci(values, n_boot=2000, seed=0):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


# ---------------------------------------------------------------------------
# Region-adaptive fusion of the ensemble members
# ---------------------------------------------------------------------------


def simplex_grid(members, step):
    """Every weight vector over `members` with entries in multiples of `step` that sum to one."""
    n = round(1 / step)
    combos = [c for c in itertools.product(range(n + 1), repeat=members) if sum(c) == n]
    return np.array(combos, dtype=np.float32) / n


def fuse(probs, weights):
    """Region-wise weighted mean of member probabilities: probs (M, 3, ...), weights (3, M)."""
    return torch.einsum("rm,mr...->r...", weights.to(probs.dtype), probs)


@torch.no_grad()
def fusion_dice_table(probs, ref, combos, thresholds):
    """Dice of every (weight vector, threshold) pair for every region of one case.

    probs (M, 3, ...) member probabilities, ref (3, ...) bool reference, combos (K, M).
    Returns an array (3, K, T).
    """
    out = np.zeros((3, len(combos), len(thresholds)))
    weights = torch.as_tensor(combos, device=probs.device, dtype=probs.dtype)
    for region in range(3):
        fused = torch.einsum("km,m...->k...", weights, probs[:, region])
        target = ref[region]
        ref_sum = int(target.sum())
        for t_index, threshold in enumerate(thresholds):
            pred = fused > threshold
            inter = (pred & target).flatten(1).sum(1).double()
            pred_sum = pred.flatten(1).sum(1).double()
            denominator = pred_sum + ref_sum
            scores = torch.where(denominator > 0, 2 * inter / denominator.clamp_min(1), torch.ones_like(denominator))
            out[region, :, t_index] = scores.cpu().numpy()
    return out


# ---------------------------------------------------------------------------
# Explainability
# ---------------------------------------------------------------------------


def seg_grad_cam(model, x, layer_name, region):
    """Seg-Grad-CAM (Vinogradova et al., AAAI 2020) for one output region of a 3D network.

    The target is the sum of the region's logits over the voxels the model assigns to the region
    (all voxels if none), the class-activation map is the ReLU of the gradient-weighted sum of the
    layer's channels, upsampled to the input and scaled to [0, 1].
    Returns the map (D, H, W) and the region probabilities (3, D, H, W).
    """
    layer = model.get_submodule(layer_name)
    store = {}

    def hook(module, inputs, output):
        store["activation"] = output
        output.register_hook(lambda grad: store.__setitem__("gradient", grad))

    handle = layer.register_forward_hook(hook)
    try:
        model.eval()
        model.zero_grad(set_to_none=True)
        with torch.enable_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=x.is_cuda):
            logits = output_list(model(x))[0].float()
            mask = (logits[:, region] > 0).float()
            if mask.sum() == 0:
                mask = torch.ones_like(mask)
            (logits[:, region] * mask).sum().backward()
        activation, gradient = store["activation"].float(), store["gradient"].float()
        cam = F.relu((gradient.mean(dim=(2, 3, 4), keepdim=True) * activation).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="trilinear", align_corners=False)[0, 0]
        cam = cam / cam.max().clamp_min(1e-8)
        return cam.detach(), torch.sigmoid(logits[0]).detach()
    finally:
        handle.remove()
        model.zero_grad(set_to_none=True)


def attribution_in_region(cam, region_mask, margin=5):
    """Share of the map's mass inside the reference region dilated by `margin` voxels, and the
    share expected by chance (the dilated region's volume fraction)."""
    mask = region_mask.float()[None, None]
    dilated = F.max_pool3d(mask, 2 * margin + 1, stride=1, padding=margin)[0, 0] > 0
    total = float(cam.sum())
    inside = float(cam[dilated].sum()) / total if total > 0 else float("nan")
    return inside, float(dilated.float().mean())

### Training worker

Written to `src/volume_worker.py` and launched once per GPU. Per job it probes the largest micro-batch that fits, trains on random patches with fp16 autocast, gradient scaling, accumulation to `BATCH`, gradient clipping and a per-step learning-rate schedule (polynomial for SGD, warm-up then cosine for AdamW), validates with sliding windows every `VAL_EVERY` epochs and keeps the best checkpoint plus a resumable last checkpoint. After training it stores the TTA probabilities of every validation, test and external case, tunes the ET size threshold on validation, scores the test and external cases, runs the sequence-drop ablation on the test cases and times inference.

In [ ]:
%%writefile /kaggle/working/src/volume_worker.py
"""Training worker for the hybrid-ensemble segmentation notebook: one process per GPU.

Each worker claims (model, seed) jobs and trains on random native-resolution patches with
checkpoints every epoch (so a later session resumes from the last epoch), keeps the epoch with
the best sliding-window validation Dice, then predicts every validation, internal test and
external case with test-time augmentation, stores the probabilities, tunes the enhancing-tumour
size threshold on the validation cases, scores the test and external cases, measures how much
each MRI sequence matters (sequence-drop ablation) and times inference.
"""

import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent))
import shared_lib  # noqa: E402
import volume_lib as lib  # noqa: E402


def seed_everything(seed, deterministic):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)


def probe_micro_batch(model, start, size, weights, device):
    """Largest micro-batch whose forward and backward pass fit; nothing is stepped."""
    model.train()
    candidate = start
    while candidate >= 1:
        try:
            x = torch.randn(candidate, 4, size, size, size, device=device)
            y = torch.zeros(candidate, 3, size, size, size, device=device)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lib.supervised_loss(model(x), y, weights)
            loss.backward()
            torch.cuda.synchronize(device)
            return candidate
        except torch.OutOfMemoryError:
            candidate //= 2
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    raise RuntimeError("Even a micro-batch of 1 does not fit in GPU memory")


def case_input(cfg, case, device):
    return torch.from_numpy(np.asarray(lib.load_image(cfg["cache_dir"], case), dtype=np.float32))[None].to(device)


@torch.no_grad()
def validate(model, val, cfg, device):
    """Mean Dice per region over the validation cases: whole cropped volumes, sliding windows, no TTA."""
    rows = []
    for case in val["case"]:
        probs, _ = lib.sliding_probs(model, case_input(cfg, case, device), cfg["patch"], cfg["sw_overlap"], False)
        pred = lib.regions_from_probs(probs[0])
        ref = torch.from_numpy(lib.load_regions(cfg["cache_dir"], case)).to(device)
        rows.append([lib.dice(pred[i], ref[i]) for i in range(3)])
    return np.array(rows).mean(axis=0)


def predict_split(model, frame, cfg, device, out_dir, reporter, label):
    """Stores the TTA probabilities of every case of a split (cases already stored are skipped)."""
    out_dir = Path(out_dir)
    start = time.perf_counter()
    for position, case in enumerate(frame["case"], start=1):
        path = out_dir / f"{case}.npz"
        if not path.exists():
            mean, std = lib.sliding_probs(model, case_input(cfg, case, device), cfg["patch"], cfg["sw_overlap"], cfg["tta"])
            wt = mean[0, 0] > 0.5
            spread = float(std[0, 0][wt].mean()) if wt.any() else 0.0
            lib.save_probs(path, mean[0], spread)
        reporter.status(job=label, step=position, total=len(frame), postfix={"split": out_dir.name})
    reporter.event(f"{label}: {len(frame)} {out_dir.name} cases predicted in {time.perf_counter() - start:.0f}s")


def tune_et_threshold(frame, cfg, prob_dir):
    """Validation ET Dice for each candidate ET size threshold."""
    scores = {threshold: [] for threshold in cfg["et_candidates"]}
    for case in frame["case"]:
        probs = torch.from_numpy(lib.load_probs(Path(prob_dir) / f"{case}.npz")[0]).float() / 255
        ref_et = torch.from_numpy(lib.load_regions(cfg["cache_dir"], case)[2])
        for threshold in scores:
            scores[threshold].append(lib.dice(lib.regions_from_probs(probs, threshold)[2], ref_et))
    return {threshold: float(np.mean(values)) for threshold, values in scores.items()}


def score_split(frame, cfg, prob_dir, et_min):
    """Per-case metrics of a split from its stored probabilities."""
    rows = []
    for case in frame.itertuples():
        stored, spread = lib.load_probs(Path(prob_dir) / f"{case.case}.npz")
        pred = lib.regions_from_probs(torch.from_numpy(stored).float() / 255, et_min).numpy()
        ref = lib.load_regions(cfg["cache_dir"], case.case)
        spacing = tuple(float(v) for v in case.spacing.split("x"))
        metrics = lib.case_metrics(pred, ref, spacing, int(case.native_voxels))
        rows.append({"case": case.case, "dataset": case.dataset, "grade": case.grade, **metrics, "tta_std_in_wt": spread})
    return pd.DataFrame(rows)


@torch.no_grad()
def modality_ablation(model, frame, cfg, device, et_min, reporter, label):
    """Dice per region of every test case with all sequences and with each sequence set to zero
    (the mean of its z-scored brain voxels), without TTA."""
    rows = []
    for position, case in enumerate(frame["case"], start=1):
        reporter.status(job=label, step=position, total=len(frame), postfix={"stage": "sequence drop"})
        x = case_input(cfg, case, device)
        ref = torch.from_numpy(lib.load_regions(cfg["cache_dir"], case)).to(device)
        for dropped in ("none", *lib.MODALITIES):
            inputs = x.clone()
            if dropped != "none":
                inputs[:, lib.MODALITIES.index(dropped)] = 0
            probs, _ = lib.sliding_probs(model, inputs, cfg["patch"], cfg["sw_overlap"], False)
            pred = lib.regions_from_probs(probs[0], et_min)
            rows.append({"case": case, "dropped": dropped, **{f"dice_{r}": lib.dice(pred[i], ref[i]) for i, r in enumerate(lib.REGIONS)}})
    return pd.DataFrame(rows)


@torch.no_grad()
def time_inference(model, shape, cfg, device, tta, iterations):
    x = torch.randn(1, 4, *shape, device=device)
    lib.sliding_probs(model, x, cfg["patch"], cfg["sw_overlap"], tta)
    torch.cuda.synchronize(device)
    timings = []
    for _ in range(iterations):
        start = time.perf_counter()
        lib.sliding_probs(model, x, cfg["patch"], cfg["sw_overlap"], tta)
        torch.cuda.synchronize(device)
        timings.append(time.perf_counter() - start)
    return float(np.mean(timings)), float(np.std(timings))


def run_job(job, cfg, cases, device, reporter):
    name, seed = job["model"], job["seed"]
    rid = lib.run_id(name, seed)
    recipe = lib.RECIPES[name]
    patch, epochs, iters = cfg["patch"], cfg["epochs"][name], cfg["iters_per_epoch"]
    seed_everything(seed, cfg["deterministic"])
    run_dir = Path(cfg["runs_dir"]) / rid
    run_dir.mkdir(parents=True, exist_ok=True)
    weights_dir = Path(cfg["weights_dir"]) / lib.slug(name)
    weights_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path = weights_dir / f"{rid}.pth", weights_dir / f"{rid}_last.pth"

    model = lib.build_model(name).to(device)
    weights = recipe["ds_weights"]
    micro = probe_micro_batch(model, recipe["micro_batch"], patch, weights, device)
    accumulation = math.ceil(cfg["batch"] / micro)
    samples_per_epoch = iters * accumulation * micro
    train = cases[cases["role"] == "train"]
    val = cases[cases["role"] == "val"]
    dataset = lib.PatchDataset(cfg["cache_dir"], train["case"], cfg["fg_path"], cfg["source"], patch, cfg["fg_probability"], seed)
    loader_kwargs = {"num_workers": cfg["loader_workers"], "pin_memory": True}
    if cfg["loader_workers"] > 0:
        loader_kwargs["prefetch_factor"] = 4
    total_steps = epochs * iters
    warmup_steps = recipe["warmup_epochs"] * iters
    if recipe["optimizer"] == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=recipe["lr"], momentum=0.99, nesterov=True, weight_decay=recipe["weight_decay"])

        def lr_factor(step):
            return (1 - min(step, total_steps - 1) / total_steps) ** 0.9

    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=recipe["lr"], weight_decay=recipe["weight_decay"])

        def lr_factor(step):
            if step < warmup_steps:
                return (step + 1) / warmup_steps
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda")
    state = {"epoch": 0, "step": 0, "history": [], "best_dice": -1.0, "best_epoch": 0, "train_seconds": 0.0, "peak_memory_gb": 0.0}
    if last_path.exists():  # resume from the end of the last finished epoch
        checkpoint = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        state = checkpoint["state"]
        torch.set_rng_state(checkpoint["cpu_rng"].cpu())
        torch.cuda.set_rng_state(checkpoint["cuda_rng"].cpu(), device)
        reporter.event(f"{rid}: resumed after epoch {state['epoch']}")
    reporter.event(
        f"{rid}: {len(train)} train / {len(val)} val cases, micro-batch {micro} x {accumulation} accumulation, "
        f"{total_steps} steps over {epochs} epochs of {samples_per_epoch} patches"
    )

    torch.cuda.reset_peak_memory_stats(device)
    while state["epoch"] < epochs:
        if time.time() > cfg["deadline"]:
            reporter.event(f"{rid}: time budget reached after epoch {state['epoch']}; resume in the next session")
            return False
        epoch = state["epoch"] + 1
        model.train()
        start, losses = time.perf_counter(), []
        sampler = range((epoch - 1) * samples_per_epoch, epoch * samples_per_epoch)
        loader = torch.utils.data.DataLoader(dataset, batch_size=micro, sampler=sampler, drop_last=True, **loader_kwargs)
        optimizer.zero_grad(set_to_none=True)
        for batch_index, (image, bits) in enumerate(loader, start=1):
            image, label = lib.augment(image.to(device, non_blocking=True), bits.to(device, non_blocking=True), patch)
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(image)
            loss = lib.supervised_loss(outputs, label, weights)
            scaler.scale(loss / accumulation).backward()
            losses.append(loss.item())
            if batch_index % accumulation == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), recipe["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                state["step"] += 1
                reporter.status(
                    job=rid,
                    step=state["step"],
                    total=total_steps,
                    postfix={"epoch": epoch, "loss": round(float(np.mean(losses)), 4), "lr": f"{scheduler.get_last_lr()[0]:.2e}", "mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2)},
                )
        del loader
        train_seconds = time.perf_counter() - start
        state["train_seconds"] += train_seconds
        state["peak_memory_gb"] = max(state["peak_memory_gb"], torch.cuda.max_memory_allocated(device) / 1e9)
        row = {
            "epoch": epoch,
            "lr": scheduler.get_last_lr()[0],
            "train_loss": float(np.mean(losses)),
            "samples_per_s": samples_per_epoch / train_seconds,
            **{f"val_dice_{region}": np.nan for region in lib.REGIONS},
            "val_dice_mean": np.nan,
        }
        improved = False
        if epoch % cfg["val_every"] == 0 or epoch == epochs:
            val_dice = validate(model, val, cfg, device)
            row.update({f"val_dice_{region}": float(value) for region, value in zip(lib.REGIONS, val_dice)})
            row["val_dice_mean"] = float(val_dice.mean())
            improved = row["val_dice_mean"] > state["best_dice"]
            if improved:
                state["best_dice"], state["best_epoch"] = row["val_dice_mean"], epoch
                torch.save({k: v.detach().float().cpu() for k, v in model.state_dict().items()}, best_path)
        row["epoch_seconds"] = time.perf_counter() - start
        state["history"].append(row)
        state["epoch"] = epoch
        pd.DataFrame(state["history"]).to_csv(run_dir / "history.csv", index=False)
        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "state": state,
                "cpu_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state(device),
            },
            last_path,
        )
        message = f"{rid} epoch {epoch}/{epochs}: train loss {row['train_loss']:.4f} | {row['samples_per_s']:.2f} patches/s"
        if not np.isnan(row["val_dice_mean"]):
            message += f" | val Dice WT {row['val_dice_WT']:.4f} TC {row['val_dice_TC']:.4f} ET {row['val_dice_ET']:.4f} mean {row['val_dice_mean']:.4f}"
        reporter.event(message + (" | best" if improved else ""))

    # Evaluation of the best epoch: probabilities of every validation, test and external case
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    model.eval()
    torch.cuda.reset_peak_memory_stats(device)
    prob_root = Path(cfg["probs_dir"]) / rid
    splits = {role: cases[cases["role"] == role] for role in ("val", "test", "external")}
    predict_seconds = {}
    for role, frame in splits.items():
        start = time.perf_counter()
        predict_split(model, frame, cfg, device, prob_root / role, reporter, f"{rid} {role}")
        predict_seconds[role] = time.perf_counter() - start
    inference_memory = torch.cuda.max_memory_allocated(device) / 1e9
    tuning = tune_et_threshold(splits["val"], cfg, prob_root / "val")
    et_min = max(tuning, key=lambda t: (tuning[t], -t))
    for role, frame in splits.items():
        score_split(frame, cfg, prob_root / role, et_min).to_csv(run_dir / f"{role}_cases.csv", index=False)
    reporter.event(f"{rid}: sequence-drop ablation on {len(splits['test'])} test cases")
    modality_ablation(model, splits["test"], cfg, device, et_min, reporter, f"{rid} ablation").to_csv(run_dir / "modality_ablation.csv", index=False)
    shape = tuple(int(np.median([int(s.split("x")[i]) for s in splits["test"]["crop_shape"]])) for i in range(3))
    latency, latency_sd = time_inference(model, shape, cfg, device, False, cfg["latency_iters"])
    latency_tta, latency_tta_sd = time_inference(model, shape, cfg, device, True, max(2, cfg["latency_iters"] // 2))
    test_scores = pd.read_csv(run_dir / "test_cases.csv")
    external_scores = pd.read_csv(run_dir / "external_cases.csv")
    summary = {
        "run_id": rid,
        "model": name,
        "family": recipe["family"],
        "seed": seed,
        "parameters": sum(p.numel() for p in model.parameters()),
        "micro_batch": micro,
        "accumulation": accumulation,
        "optimizer_steps": state["step"],
        "epochs_run": state["epoch"],
        "patches_seen": state["epoch"] * samples_per_epoch,
        "best_epoch": state["best_epoch"],
        "best_val_dice": state["best_dice"],
        "et_min_voxels": et_min,
        "et_tuning": {str(k): v for k, v in tuning.items()},
        "train_seconds": state["train_seconds"],
        "predict_seconds": predict_seconds,
        "peak_train_memory_gb": state["peak_memory_gb"],
        "peak_inference_memory_gb": inference_memory,
        "latency_shape": "x".join(map(str, shape)),
        "latency_s": latency,
        "latency_s_sd": latency_sd,
        "latency_tta_s": latency_tta,
        "latency_tta_s_sd": latency_tta_sd,
        "test_dice_mean": float(test_scores["dice_mean"].mean()),
        "external_dice_mean": float(external_scores["dice_mean"].mean()) if len(external_scores) else float("nan"),
        "gpu": torch.cuda.get_device_name(device),
    }
    shared_lib.write_json(run_dir / "summary.json", summary)
    last_path.unlink(missing_ok=True)  # optimiser state is no longer needed
    reporter.event(f"{rid} done: test mean Dice {summary['test_dice_mean']:.4f}, external {summary['external_dice_mean']:.4f}, ET size threshold {et_min} voxels")
    del model, optimizer
    torch.cuda.empty_cache()
    return True


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--gpu", type=int, required=True)
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    device = torch.device("cuda:0")  # CUDA_VISIBLE_DEVICES selects the physical GPU
    torch.cuda.set_device(device)
    reporter = shared_lib.Reporter(cfg["log_dir"], args.gpu)
    cases = pd.read_csv(cfg["case_table"])
    claims = Path(cfg["claims_dir"])
    for job in cfg["jobs"]:
        rid = lib.run_id(job["model"], job["seed"])
        if (Path(cfg["runs_dir"]) / rid / "summary.json").exists():
            continue
        if time.time() > cfg["deadline"]:
            reporter.event(f"Time budget reached; {rid} and later jobs left for the next session")
            break
        try:
            os.close(os.open(claims / rid, os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        except FileExistsError:
            continue
        run_job(job, cfg, cases, device, reporter)
    reporter.status(job=None)


if __name__ == "__main__":
    main()

### Worker launcher, monitor and shared helpers

Written to `src/shared_lib.py`: starts one worker per GPU (each sees one device through `CUDA_VISIBLE_DEVICES`), polls their status files to draw an overall jobs bar, one step bar per GPU and a line per finished epoch, and stops the cell with the end of a worker's log if it fails. Also holds the workers' progress reporter and the Holm correction.

In [ ]:
%%writefile /kaggle/working/src/shared_lib.py
"""Helpers shared by the unified notebooks: worker launch and live monitoring, progress
reporting from the workers, JSON files, and the Holm correction.

Each notebook writes this file to /kaggle/working/src; training runs in one subprocess per GPU
and the notebook turns the workers' status files into live progress bars.
"""

import json
import os
import sys
import time
from pathlib import Path

import numpy as np


def read_json(path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return default


def write_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, default=float))
    os.replace(tmp, path)


def launch_workers(script, config_path, log_dir, gpu_count):
    import subprocess

    processes = []
    for gpu in range(gpu_count):
        env = os.environ.copy()
        env.update(
            CUDA_VISIBLE_DEVICES=str(gpu),
            PYTHONUNBUFFERED="1",
            CUBLAS_WORKSPACE_CONFIG=":4096:8",
            OMP_NUM_THREADS=str(max(1, (os.cpu_count() or 2) // gpu_count)),
        )
        log = open(Path(log_dir) / f"worker_gpu{gpu}.log", "a")
        command = [sys.executable, str(script), "--config", str(config_path), "--gpu", str(gpu)]
        processes.append((subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    return processes


def monitor_workers(processes, log_dir, total_jobs, finished_jobs, poll_seconds=3):
    """Live progress: one bar for all jobs, one bar per GPU, epoch lines as they arrive."""
    from tqdm.auto import tqdm

    log_dir = Path(log_dir)
    overall = tqdm(total=total_jobs, desc="Jobs", unit="job", position=0)
    overall.update(finished_jobs())
    bars = [
        tqdm(total=1, desc=f"GPU {gpu}: waiting", unit="step", position=gpu + 1, leave=True)
        for gpu in range(len(processes))
    ]
    offsets = [0] * len(processes)
    current_job = [None] * len(processes)
    while True:
        for gpu, bar in enumerate(bars):
            status = read_json(log_dir / f"status_gpu{gpu}.json", {})
            if status.get("job"):
                if status["job"] != current_job[gpu]:
                    current_job[gpu] = status["job"]
                    bar.reset(total=max(1, status.get("total", 1)))
                    bar.set_description(f"GPU {gpu}: {status['job']}")
                bar.n = min(status.get("step", 0), bar.total)
                bar.set_postfix(status.get("postfix", {}), refresh=False)
                bar.refresh()
            events = log_dir / f"events_gpu{gpu}.jsonl"
            if events.exists():
                with open(events) as handle:
                    handle.seek(offsets[gpu])
                    for line in handle:
                        tqdm.write(json.loads(line)["message"])
                    offsets[gpu] = handle.tell()
        done = finished_jobs()
        overall.update(done - overall.n)
        if all(process.poll() is not None for process, _ in processes):
            break
        time.sleep(poll_seconds)
    for bar in bars:
        bar.close()
    overall.close()
    failures = []
    for gpu, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            tail = (log_dir / f"worker_gpu{gpu}.log").read_text().splitlines()[-40:]
            failures.append(f"GPU {gpu} worker exited with {process.returncode}:\n" + "\n".join(tail))
    if failures:
        raise RuntimeError("\n\n".join(failures))


class Reporter:
    """Status file (polled by the notebook) and epoch events written by a worker."""

    def __init__(self, log_dir, gpu):
        self.status_path = Path(log_dir) / f"status_gpu{gpu}.json"
        self.events_path = Path(log_dir) / f"events_gpu{gpu}.jsonl"
        self.gpu = gpu

    def status(self, **fields):
        write_json(self.status_path, fields)

    def event(self, message, **fields):
        with open(self.events_path, "a") as handle:
            handle.write(json.dumps({"time": time.time(), "message": f"[GPU {self.gpu}] {message}", **fields}) + "\n")
        print(message, flush=True)


def holm(p_values):
    """Holm-Bonferroni adjusted p-values, same order as the input."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted

The libraries are imported from `src/`, and every member is built on the CPU to count its parameters, then moved to GPU 0 for one forward pass that measures GFLOPs per 128³ patch. The meta device is not used because MONAI's Swin UNETR calls `.item()` while it is built. GPU memory is released before training. Expected output: one row per model.

In [ ]:
sys.path.insert(0, str(SRC))
import shared_lib
import volume_lib as lib

importlib.reload(lib)
importlib.reload(shared_lib)
from torch.utils.flop_counter import FlopCounterMode

profiles = []
for name in MODELS:
    model = lib.build_model(name)
    parameters = sum(p.numel() for p in model.parameters())
    model = model.to(DEVICE).eval()
    try:
        with torch.no_grad(), FlopCounterMode(display=False) as counter:
            model(torch.zeros(1, 4, PATCH, PATCH, PATCH, device=DEVICE))
        gflops = counter.get_total_flops() / 1e9
    except Exception as exc:  # a failed count only loses the FLOP estimate
        log(f"{name}: FLOPs not estimated ({type(exc).__name__})")
        gflops = float("nan")
    profiles.append(
        {
            "model": name,
            "family": lib.RECIPES[name]["family"],
            "parameters_m": parameters / 1e6,
            "gflops_per_patch": gflops,
            **{k: v for k, v in lib.RECIPES[name].items() if k != "family"},
        }
    )
    del model
    torch.cuda.empty_cache()
PROFILES = pd.DataFrame(profiles).set_index("model")
save_table(PROFILES, "model_profiles", index=True)

## 2. Data

### Find the cases

Locates both datasets under `/kaggle/input` (old and new mount layouts), lists every case folder with its four sequences and its mask, and joins the BraTS 2020 tumour grade (HGG or LGG) from `name_mapping.csv`. BraTS-Africa records no grade. Expected output: case counts per dataset and grade.

In [ ]:
def find_dataset_root(slug):
    for candidate in [INPUT / slug, *INPUT.glob(f"*/{slug}"), *INPUT.glob(f"*/*/{slug}")]:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Dataset '{slug}' is not attached: add it with 'Add Input' in the notebook sidebar."
    )


def list_cases(root, pattern, dataset, et_label):
    rows, seen = [], set()
    for case_dir in sorted(
        p for p in root.rglob(pattern) if p.is_dir() and ".nii" not in p.name.lower()
    ):
        if case_dir.name in seen:
            continue
        seen.add(case_dir.name)
        files = lib.case_files(case_dir)
        rows.append(
            {
                "case": case_dir.name,
                "dataset": dataset,
                "et_label": et_label,
                **{key: str(files.get(key, "")) for key in (*lib.MODALITIES, "seg")},
            }
        )
    return pd.DataFrame(rows)


TRAIN_ROOT = next(find_dataset_root(DATASET_SLUG).rglob("MICCAI_BraTS2020_TrainingData"))
internal = list_cases(TRAIN_ROOT, "BraTS20_Training_*", "BraTS 2020", 4)
mapping_path = next(TRAIN_ROOT.glob("name_mapping*.csv"), None)
grades = {}
if mapping_path is not None:
    mapping = pd.read_csv(mapping_path)
    grades = dict(zip(mapping["BraTS_2020_subject_ID"], mapping["Grade"]))
internal["grade"] = internal["case"].map(grades).fillna("unknown")
external = list_cases(find_dataset_root(EXTERNAL_SLUG), "BraTS-SSA-*", "BraTS-Africa", 3)
external["grade"] = "SSA"
CASES = pd.concat([internal, external], ignore_index=True)
complete = CASES[[*lib.MODALITIES, "seg"]].ne("").all(axis=1)
if (~complete).any():
    log(f"Cases missing a file (excluded): {CASES.loc[~complete, 'case'].tolist()}")
CASES = CASES[complete].reset_index(drop=True)
assert (CASES["dataset"] == "BraTS-Africa").any(), "No complete BraTS-Africa case was found."
display(pd.crosstab(CASES["dataset"], CASES["grade"], margins=True, margins_name="Total"))

### Preprocess and cache

One parallel pass (all CPU cores) crops every case to the bounding box of its brain, z-scores each sequence over its non-zero voxels and stores float16 images and the 3-bit region mask at the native 1 mm resolution in `/tmp` (no resampling, unlike the earlier 128³ pipeline). Tumour voxel coordinates are kept for patch oversampling. The checks confirm the label convention of each dataset and that no reference voxel lies outside the crop, so metrics on the cropped grid equal those on the native grid. Expected output: a progress bar, the checks and the cache size.

In [ ]:
jobs = [
    (
        {key: row[key] for key in (*lib.MODALITIES, "seg")},
        row["et_label"],
        str(CASE_CACHE / row["case"]),
        FG_SAMPLES,
    )
    for row in CASES.to_dict("records")
]
info_rows, foreground = [], {}
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
    results = pool.map(lib.preprocess_case, jobs, chunksize=2)
    for case, info in zip(
        CASES["case"], tqdm(results, total=len(jobs), desc="Preprocessing cases")
    ):
        foreground.update({f"{case}/{region}": info["fg"][region] for region in REGIONS})
        info_rows.append(
            {
                "shape": "x".join(map(str, info["shape"])),
                "spacing": "x".join(f"{s:g}" for s in info["spacing"]),
                "native_voxels": int(np.prod(info["shape"])),
                "crop_shape": "x".join(map(str, info["crop_shape"])),
                "box_low": "x".join(map(str, info["box_low"])),
                "label_outside_crop": info["label_outside_crop"],
                "labels": " ".join(map(str, info["labels_present"])),
                **{f"volume_cm3_{r}": v for r, v in zip(REGIONS, info["volumes_cm3"])},
            }
        )
FG_PATH = CACHE / "fg_coords.npz"
np.savez(FG_PATH, **foreground)
CASES = pd.concat([CASES, pd.DataFrame(info_rows)], axis=1)
expected_labels = {"BraTS 2020": {0, 1, 2, 4}, "BraTS-Africa": {0, 1, 2, 3}}
for dataset, allowed in expected_labels.items():
    found = set().union(
        *(set(map(int, s.split())) for s in CASES.loc[CASES["dataset"] == dataset, "labels"])
    )
    assert found <= allowed, f"{dataset}: unexpected labels {sorted(found - allowed)}"
outside = int(CASES["label_outside_crop"].sum())
log(f"Reference voxels outside the brain crop: {outside}")
cache_gb = sum(p.stat().st_size for p in CASE_CACHE.iterdir()) / 1e9
log(f"Cached {len(CASES)} cases in {time.perf_counter() - start:.0f}s ({cache_gb:.2f} GB)")
display(CASES.groupby("dataset")[["shape", "spacing"]].value_counts().rename("cases").to_frame())
crop_dims = CASES["crop_shape"].str.split("x", expand=True).astype(int)
display(crop_dims.describe().round(1).rename(columns={0: "crop D", 1: "crop H", 2: "crop W"}))

### Dataset overview

Region volumes per cohort (internal HGG, internal LGG, external) and how often ET is absent, which drives the ET results; one internal and one external case with their four sequences and region masks. Expected output: a table and two figures.

In [ ]:
CASES["cohort"] = np.where(
    CASES["dataset"] == "BraTS-Africa", "BraTS-Africa", "BraTS 2020 " + CASES["grade"]
)
volume_stats = CASES.groupby("cohort")[[f"volume_cm3_{r}" for r in REGIONS]].describe().T
save_table(volume_stats.round(2), "region_volumes_by_cohort", index=True)
no_et = (
    CASES.assign(no_et=CASES["volume_cm3_ET"] == 0).groupby("cohort")["no_et"].agg(["sum", "mean"])
)
save_table(
    no_et.rename(columns={"sum": "cases_without_ET", "mean": "share"}).round(3),
    "cases_without_et",
    index=True,
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, region in zip(axes, REGIONS):
    for cohort, sub in CASES.groupby("cohort"):
        ax.hist(
            sub[f"volume_cm3_{region}"],
            bins=30,
            alpha=0.55,
            density=True,
            label=f"{cohort} ({len(sub)})",
        )
    ax.set(title=f"{region} volume per case", xlabel="cm³", ylabel="Density")
    ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "01_region_volumes")

fig, axes = plt.subplots(2, 5, figsize=(19, 8))
for row, dataset in enumerate(("BraTS 2020", "BraTS-Africa")):
    sub = CASES[CASES["dataset"] == dataset]
    example = sub.loc[(sub["volume_cm3_ET"] - sub["volume_cm3_ET"].median()).abs().idxmin()]
    volume = np.asarray(lib.load_image(CASE_CACHE, example["case"]), dtype=np.float32)
    regions = lib.load_regions(CASE_CACHE, example["case"])
    z = int(np.argmax(regions[0].sum(axis=(0, 1))))
    for col, modality in enumerate(lib.MODALITIES):
        axes[row, col].imshow(np.rot90(volume[col, :, :, z]), cmap="gray")
        axes[row, col].set_title(f"{modality.upper()} ({dataset})", fontsize=9)
    axes[row, 4].imshow(np.rot90(volume[0, :, :, z]), cmap="gray")
    colour = np.zeros((*volume.shape[1:3], 4))
    for index, rgba in enumerate(
        [(0.0, 0.8, 0.0, 0.45), (1.0, 0.85, 0.0, 0.55), (1.0, 0.0, 0.0, 0.65)]
    ):
        colour[regions[index][:, :, z]] = rgba
    axes[row, 4].imshow(np.rot90(colour, axes=(0, 1)))
    axes[row, 4].set_title(f"{example['case']}\nWT green, TC yellow, ET red", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle(
    "Median-ET cases of each dataset, axial slice with the largest whole tumour (brain crop, native 1 mm)"
)
fig.tight_layout()
save_fig(fig, "02_modalities_and_masks")

### Split

The BraTS 2020 validation and test patients are exactly those of the earlier six-network benchmark (grade-stratified 257/56/56), fixed by ID rather than recomputed, so results are paired by case across both studies. Every BraTS-Africa case is external: used only for the final evaluation. Expected output: cases per role and grade.

In [ ]:
def training_ids(text):
    return {f"BraTS20_Training_{number}" for number in text.split()}


val_names, test_names = training_ids(VAL_IDS), training_ids(TEST_IDS)
missing = (val_names | test_names) - set(CASES.loc[CASES["dataset"] == "BraTS 2020", "case"])
assert not missing, f"Fixed split cases not found: {sorted(missing)}"
CASES["role"] = np.select(
    [
        CASES["dataset"] == "BraTS-Africa",
        CASES["case"].isin(test_names),
        CASES["case"].isin(val_names),
    ],
    ["external", "test", "val"],
    "train",
)
CASE_TABLE = METRICS / "cases.csv"
CASES.to_csv(CASE_TABLE, index=False)
split_table = pd.crosstab(
    CASES["role"], CASES["grade"], margins=True, margins_name="Total"
).reindex(["train", "val", "test", "external", "Total"])
save_table(split_table, "split", index=True)
VAL_CASES = CASES[CASES["role"] == "val"].reset_index(drop=True)
EVAL_CASES = {role: CASES[CASES["role"] == role].reset_index(drop=True) for role in ROLES}

### Augmentation preview

One tumour-centred training cube, its plain centre crop, and three draws of the GPU augmentation (rotation and zoom with probability 0.2 each, mirroring, noise, brightness, contrast, gamma). Expected output: one figure.

In [ ]:
preview = lib.PatchDataset(
    CASE_CACHE, CASES.loc[CASES["role"] == "train", "case"], FG_PATH, SOURCE, PATCH, 1.0, SPLIT_SEED
)
cube, cube_bits = preview[0]
cube, cube_bits = cube[None].to(DEVICE), cube_bits[None].to(DEVICE)
offset = (SOURCE - PATCH) // 2
centre = slice(offset, offset + PATCH)
torch.manual_seed(SPLIT_SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    if col == 0:
        aug_image = cube[:, :, centre, centre, centre]
        aug_label = ((cube_bits[:, None, centre, centre, centre] & 1) > 0).float()
    else:
        aug_image, aug_label = lib.augment(cube, cube_bits, PATCH, p_rotate=0.5, p_zoom=0.5)
    z = PATCH // 2
    axes[0, col].imshow(np.rot90(aug_image[0, 0, :, :, z].cpu().numpy()), cmap="gray")
    axes[1, col].imshow(np.rot90(aug_label[0, 0, :, :, z].cpu().numpy()), cmap="gray")
    axes[0, col].set_title(
        "Centre crop (FLAIR)" if col == 0 else f"Augmented {col} (p = 0.5 for the preview)"
    )
    axes[1, col].set_title("WT mask")
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "03_augmentation")

## 3. Training

Builds the job list (every member x seed, largest first so the transformer and the CNNs run on different GPUs), restores earlier runs from `RESUME_DIR` (after checking the split is identical), writes the worker configuration and launches one worker per GPU. The cell follows them live: an overall jobs bar, a step bar per GPU with loss, learning rate and peak memory, and one line per epoch with throughput and, every `VAL_EVERY` epochs, validation Dice per region. Prediction, scoring and the sequence-drop ablation of each member also report progress. Jobs unfinished at `TRAINING_BUDGET_HOURS` stop cleanly at an epoch boundary. Expected output: live progress.

In [ ]:
JOBS = [{"model": m, "seed": s} for m in MODELS for s in SEEDS]
JOBS.sort(key=lambda job: -PROFILES.loc[job["model"], "parameters_m"])
if RESUME_DIR is not None:
    previous = pd.read_csv(Path(RESUME_DIR) / "metrics" / "cases.csv")
    assert previous[["case", "role"]].equals(
        CASES[["case", "role"]]
    ), "RESUME_DIR was produced with a different split."
    for sub in ("metrics/runs", "predictions", "weights"):
        if (Path(RESUME_DIR) / sub).exists():
            shutil.copytree(Path(RESUME_DIR) / sub, WORK / sub, dirs_exist_ok=True)
    log(
        f"Restored {len(list(RUNS.glob('*/summary.json')))} finished runs and {len(list(WEIGHTS.glob('*/*_last.pth')))} resumable runs"
    )

shutil.rmtree(CLAIMS, ignore_errors=True)
CLAIMS.mkdir(parents=True)
for path in LOGS.glob("status_gpu*.json"):
    path.unlink()
worker_config = {
    "patch": PATCH,
    "source": SOURCE,
    "epochs": EPOCHS,
    "iters_per_epoch": ITERS_PER_EPOCH,
    "batch": BATCH,
    "fg_probability": FG_PROBABILITY,
    "val_every": VAL_EVERY,
    "sw_overlap": SW_OVERLAP,
    "deterministic": DETERMINISTIC,
    "tta": TTA,
    "et_candidates": ET_CANDIDATES,
    "loader_workers": LOADER_WORKERS,
    "latency_iters": LATENCY_ITERS,
    "cache_dir": str(CASE_CACHE),
    "fg_path": str(FG_PATH),
    "case_table": str(CASE_TABLE),
    "runs_dir": str(RUNS),
    "weights_dir": str(WEIGHTS),
    "probs_dir": str(PREDICTIONS),
    "log_dir": str(LOGS),
    "claims_dir": str(CLAIMS),
    "deadline": time.time() + TRAINING_BUDGET_HOURS * 3600,
    "jobs": JOBS,
}
CONFIG_PATH = METRICS / "worker_config.json"
shared_lib.write_json(CONFIG_PATH, worker_config)


def finished_jobs():
    return sum((RUNS / lib.run_id(j["model"], j["seed"]) / "summary.json").exists() for j in JOBS)


log(f"{len(JOBS)} jobs, {finished_jobs()} already finished; launching {GPU_COUNT} workers")
training_start = time.perf_counter()
workers = shared_lib.launch_workers(SRC / "volume_worker.py", CONFIG_PATH, LOGS, GPU_COUNT)
shared_lib.monitor_workers(workers, LOGS, len(JOBS), finished_jobs)
TRAINING_WALL_SECONDS = time.perf_counter() - training_start
pending = [
    lib.run_id(j["model"], j["seed"])
    for j in JOBS
    if not (RUNS / lib.run_id(j["model"], j["seed"]) / "summary.json").exists()
]
log(
    f"Training stage finished in {TRAINING_WALL_SECONDS / 60:.1f} min; pending runs: {pending or 'none'}"
)

## 4. Internal validation of the members

### Training curves

Training loss, validation Dice per region (sliding window, no TTA) and training throughput for every run with a history (finished or paused). Expected output: one figure with a row per member.

In [ ]:
SUMMARIES = pd.DataFrame([shared_lib.read_json(p) for p in sorted(RUNS.glob("*/summary.json"))])
assert len(SUMMARIES), "No finished runs yet: rerun with RESUME_DIR after the next session."
models_done = [m for m in MODELS if m in set(SUMMARIES["model"])]
if len(models_done) < len(MODELS):
    log(
        f"Members without a finished run (left out below): {[m for m in MODELS if m not in models_done]}"
    )
HISTORIES = {path.parent.name: pd.read_csv(path) for path in RUNS.glob("*/history.csv")}
models_with_history = [
    m for m in MODELS if any(k.startswith(lib.slug(m) + "_seed") for k in HISTORIES)
]
fig, axes = plt.subplots(
    len(models_with_history), 3, figsize=(18, 3.2 * len(models_with_history)), squeeze=False
)
for row, name in enumerate(models_with_history):
    for rid, history in HISTORIES.items():
        if not rid.startswith(lib.slug(name) + "_seed"):
            continue
        seed = rid.split("_")[-1]
        axes[row, 0].plot(history["epoch"], history["train_loss"], label=f"train ({seed})")
        validated = history.dropna(subset=["val_dice_mean"])
        for region in REGIONS:
            axes[row, 1].plot(
                validated["epoch"],
                validated[f"val_dice_{region}"],
                marker="o",
                label=f"{region} ({seed})",
            )
        axes[row, 1].plot(
            validated["epoch"], validated["val_dice_mean"], color="black", linewidth=2, label="mean"
        )
        axes[row, 2].plot(history["epoch"], history["samples_per_s"], label=seed)
    axes[row, 0].set_ylabel(name)
    for col in range(3):
        axes[row, col].legend(fontsize=7, ncol=2)
axes[0, 0].set_title("Training loss (Dice + BCE, deep supervision)")
axes[0, 1].set_title("Validation Dice (sliding window, no TTA)")
axes[0, 2].set_title("Training throughput (patches/s)")
fig.tight_layout()
save_fig(fig, "04_training_curves")

### Member results on the internal test set

Per-case metrics of every member (averaged over seeds per case) on the 56 internal test patients, summarised as the mean over cases with a bootstrap 95% CI of the mean Dice, plus HD95 (median, because a missed or spurious region costs the fixed BraTS value of 373.13 mm), sensitivity and specificity per region. `summarise` is reused for the ensembles and the external set. Expected output: the member table.

In [ ]:
MEMBER_KIND = {m: lib.RECIPES[m]["family"] for m in MODELS}


def member_cases(role):
    frames = [
        pd.read_csv(RUNS / s["run_id"] / f"{role}_cases.csv").assign(
            model=s["model"], seed=s["seed"]
        )
        for s in SUMMARIES.to_dict("records")
    ]
    frame = pd.concat(frames, ignore_index=True)
    return (
        frame.groupby(["model", "case", "dataset", "grade"], as_index=False)
        .mean(numeric_only=True)
        .drop(columns="seed")
    )


def summarise(per_case, order):
    rows = []
    for name in order:
        sub = per_case[per_case["model"] == name]
        if sub.empty:
            continue
        low, high = lib.bootstrap_mean_ci(sub["dice_mean"], BOOTSTRAP_SAMPLES, SPLIT_SEED)
        row = {
            "model": name,
            "kind": MEMBER_KIND.get(name, "Ensemble"),
            "cases": len(sub),
            "dice_mean": sub["dice_mean"].mean(),
            "dice_mean_ci": f"[{low:.4f}, {high:.4f}]",
        }
        for region in REGIONS:
            row[f"dice_{region}"] = sub[f"dice_{region}"].mean()
            row[f"dice_{region}_sd"] = sub[f"dice_{region}"].std()
            row[f"hd95_{region}"] = sub[f"hd95_{region}"].median()
            row[f"sensitivity_{region}"] = sub[f"sensitivity_{region}"].mean()
            row[f"specificity_{region}"] = sub[f"specificity_{region}"].mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")


MEMBER_CASES = {role: member_cases(role) for role in ("val", *ROLES)}
save_table(summarise(MEMBER_CASES["test"], models_done).round(4), "member_test_results", index=True)

## 5. Region-adaptive hybrid ensemble

### Fusion search on the validation cases

For every region separately, every convex weight vector over the members on a `FUSION_STEP` grid (66 vectors for three members, plus the exact equal-weight vector) is combined with every threshold in `FUSION_THRESHOLDS`; the pair with the highest mean validation Dice is kept. The search runs on the GPU from the stored TTA probabilities and never sees a test or external case. The region hierarchy (ET inside TC inside WT) and the ET size filter are applied afterwards. Expected output: the chosen weights and thresholds per region next to the plain mean ensemble and the best single member, and a figure.

In [ ]:
FUSION_MEMBERS = models_done
assert len(FUSION_MEMBERS) >= 2, "The ensemble needs at least two finished members."
MEMBER_RUNS = {m: SUMMARIES.loc[SUMMARIES["model"] == m, "run_id"].tolist() for m in FUSION_MEMBERS}
MEMBER_ET_MIN = {
    m: int(SUMMARIES.loc[SUMMARIES["model"] == m, "et_min_voxels"].median()) for m in FUSION_MEMBERS
}
UNIFORM = np.full(len(FUSION_MEMBERS), 1 / len(FUSION_MEMBERS), dtype=np.float32)
COMBOS = np.vstack(
    [lib.simplex_grid(len(FUSION_MEMBERS), FUSION_STEP), UNIFORM]
)  # exact mean ensemble last
SINGLE = {
    m: int(np.argmax(COMBOS[:, j])) for j, m in enumerate(FUSION_MEMBERS)
}  # weight 1 on one member
assert 0.5 in FUSION_THRESHOLDS, "FUSION_THRESHOLDS must include 0.5 for the ablations"


def member_probs(name, role, case):
    """Probabilities (3, D, H, W) in [0, 1] of one member, averaged over its seeds."""
    arrays = [
        lib.load_probs(PREDICTIONS / rid / role / f"{case}.npz")[0].astype(np.float32)
        for rid in MEMBER_RUNS[name]
    ]
    return np.mean(arrays, axis=0) / 255


def stacked_probs(role, case):
    return torch.from_numpy(np.stack([member_probs(m, role, case) for m in FUSION_MEMBERS])).to(
        DEVICE, torch.float16
    )


search = np.zeros((3, len(COMBOS), len(FUSION_THRESHOLDS)))
for case in tqdm(VAL_CASES["case"], desc="Fusion search (validation)"):
    reference = torch.from_numpy(lib.load_regions(CASE_CACHE, case)).to(DEVICE)
    search += lib.fusion_dice_table(
        stacked_probs("val", case), reference, COMBOS, FUSION_THRESHOLDS
    )
search /= len(VAL_CASES)
uniform = len(COMBOS) - 1
half = FUSION_THRESHOLDS.index(0.5)
fusion_rows, best_weights, best_thresholds, weights_at_half = [], [], [], []
for r, region in enumerate(REGIONS):
    k, t = np.unravel_index(np.argmax(search[r]), search[r].shape)
    k_half = int(np.argmax(search[r][:, half]))
    best_weights.append(COMBOS[k])
    best_thresholds.append(FUSION_THRESHOLDS[t])
    weights_at_half.append(COMBOS[k_half])
    singles = {m: search[r][SINGLE[m], half] for m in FUSION_MEMBERS}
    fusion_rows.append(
        {
            "region": region,
            **{f"weight_{m}": float(w) for m, w in zip(FUSION_MEMBERS, COMBOS[k])},
            "threshold": FUSION_THRESHOLDS[t],
            "val_dice_region_adaptive": search[r][k, t],
            "val_dice_weights_only": search[r][k_half, half],
            "val_dice_mean_ensemble": search[r][uniform, half],
            "best_single_member": max(singles, key=singles.get),
            "val_dice_best_single": max(singles.values()),
        }
    )
FUSION = pd.DataFrame(fusion_rows).set_index("region")
save_table(FUSION.round(4), "fusion_weights", index=True)

PROPOSED = "Region-adaptive hybrid ensemble (proposed)"
cnn_members = [m for m in FUSION_MEMBERS if MEMBER_KIND[m] == "CNN"]
ENSEMBLES = {
    "Mean ensemble (all members)": (np.tile(UNIFORM, (3, 1)), [0.5] * 3),
    "Region-adaptive weights (threshold 0.5)": (np.stack(weights_at_half), [0.5] * 3),
    PROPOSED: (np.stack(best_weights), best_thresholds),
}
if 1 < len(cnn_members) < len(FUSION_MEMBERS):
    cnn_weights = np.array(
        [1 / len(cnn_members) if m in cnn_members else 0.0 for m in FUSION_MEMBERS]
    )
    ENSEMBLES = {"CNN mean ensemble": (np.tile(cnn_weights, (3, 1)), [0.5] * 3), **ENSEMBLES}
ROW_ORDER = FUSION_MEMBERS + list(ENSEMBLES)
COLOURS = dict(zip(ROW_ORDER, plt.get_cmap("tab10").colors))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
bottom = np.zeros(3)
for i, name in enumerate(FUSION_MEMBERS):
    values = np.array([w[i] for w in best_weights])
    axes[0].bar(REGIONS, values, bottom=bottom, color=COLOURS[name], label=name)
    bottom += values
axes[0].set(title="Region-adaptive weights (validation-selected)", ylabel="Weight", ylim=(0, 1))
axes[0].legend(fontsize=8)
for r, region in enumerate(REGIONS):
    k = int(np.argmax(search[r].max(axis=1)))
    axes[1].plot(FUSION_THRESHOLDS, search[r][k], marker="o", label=f"{region} (best weights)")
    axes[1].plot(
        FUSION_THRESHOLDS,
        search[r][uniform],
        linestyle="--",
        alpha=0.6,
        label=f"{region} (mean ensemble)",
    )
axes[1].set(
    title="Validation Dice against the region threshold",
    xlabel="Threshold",
    ylabel="Mean validation Dice",
)
axes[1].legend(fontsize=8, ncol=2)
fig.tight_layout()
save_fig(fig, "05_fusion_search")

### Ensembles on the internal test and external sets

Each ensemble gets its own ET size threshold from the validation cases, then every internal test and external case is scored with the same native-resolution metrics as the members (HD95 on all CPU cores). The combined tables put the members and the ensembles side by side. Expected output: progress bars and the internal results table (the external table follows in section 6).

In [ ]:
def fused_probs(name, role, case):
    weights, _ = ENSEMBLES[name]
    return lib.fuse(stacked_probs(role, case), torch.as_tensor(weights, device=DEVICE)).float()


ET_MIN = {}
for name, (_, thresholds) in ENSEMBLES.items():
    scores = {t: [] for t in ET_CANDIDATES}
    for case in VAL_CASES["case"]:
        probs = fused_probs(name, "val", case)
        ref_et = torch.from_numpy(lib.load_regions(CASE_CACHE, case)[2]).to(DEVICE)
        for t in ET_CANDIDATES:
            scores[t].append(lib.dice(lib.regions_from_probs(probs, t, thresholds)[2], ref_et))
    means = {t: float(np.mean(v)) for t, v in scores.items()}
    ET_MIN[name] = max(means, key=lambda t: (means[t], -t))
log(f"ET size thresholds (voxels): {ET_MIN}")


def score_case(args):
    pred, case, cache_dir, spacing, voxels = args
    return lib.case_metrics(pred, lib.load_regions(cache_dir, case), spacing, voxels)


def run_scoring(tasks, total, desc):
    """Scores cases on all CPU cores while the GPU keeps producing the next predictions."""
    results, pending = [], []
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
        for task in tqdm(tasks, total=total, desc=desc):
            pending.append(pool.submit(score_case, task))
            if len(pending) >= 2 * os.cpu_count():
                results.append(pending.pop(0).result())
        results += [future.result() for future in pending]
    return results


def ensemble_tasks(name, frame, role):
    _, thresholds = ENSEMBLES[name]
    for row in frame.itertuples():
        pred = (
            lib.regions_from_probs(fused_probs(name, role, row.case), ET_MIN[name], thresholds)
            .cpu()
            .numpy()
        )
        yield pred, row.case, CASE_CACHE, tuple(float(v) for v in row.spacing.split("x")), int(
            row.native_voxels
        )


ALL_CASES, RESULTS = {}, {}
for role, frame in EVAL_CASES.items():
    frames = [MEMBER_CASES[role]]
    for name in ENSEMBLES:
        metrics = run_scoring(ensemble_tasks(name, frame, role), len(frame), f"{name} ({role})")
        frames.append(
            pd.DataFrame(metrics).assign(
                model=name,
                case=frame["case"].values,
                dataset=frame["dataset"].values,
                grade=frame["grade"].values,
            )
        )
    ALL_CASES[role] = pd.concat(frames, ignore_index=True)
    ALL_CASES[role].to_csv(METRICS / f"{role}_per_case.csv", index=False)
    RESULTS[role] = summarise(ALL_CASES[role], ROW_ORDER)
    save_table(RESULTS[role].round(4), f"{role}_results", index=True, show=role == "test")

### Statistical comparison

The proposed ensemble against every member and every other ensemble, separately on the internal test and external sets: Wilcoxon signed-rank tests on per-case mean Dice (paired by case), Holm-corrected within each set, with the mean paired difference and its bootstrap 95% CI. Expected output: a table and a forest plot.

In [ ]:
test_rows = []
for role, label in ROLES.items():
    wide = ALL_CASES[role].pivot(index="case", columns="model", values="dice_mean")
    others = [m for m in ROW_ORDER if m != PROPOSED and m in wide]
    raw = []
    for other in others:
        diff = wide[PROPOSED] - wide[other]
        raw.append(1.0 if np.allclose(diff, 0) else wilcoxon(wide[PROPOSED], wide[other]).pvalue)
        low, high = lib.bootstrap_mean_ci(diff, BOOTSTRAP_SAMPLES, SPLIT_SEED)
        test_rows.append(
            {
                "set": label,
                "compared_with": other,
                "mean_difference": diff.mean(),
                "ci_low": low,
                "ci_high": high,
                "median_difference": diff.median(),
                "p_raw": raw[-1],
            }
        )
    for row, p_adj in zip(test_rows[-len(others) :], shared_lib.holm(raw)):
        row["p_holm"] = p_adj
TESTS = pd.DataFrame(test_rows)
save_table(TESTS.round(5), "wilcoxon_tests")
fig, axes = plt.subplots(1, len(ROLES), figsize=(15, 0.5 * len(ROW_ORDER) + 2), sharey=True)
for ax, label in zip(np.atleast_1d(axes), ROLES.values()):
    sub = TESTS[TESTS["set"] == label].reset_index(drop=True)
    y = np.arange(len(sub))
    ax.errorbar(
        sub["mean_difference"],
        y,
        xerr=[sub["mean_difference"] - sub["ci_low"], sub["ci_high"] - sub["mean_difference"]],
        fmt="o",
        capsize=3,
    )
    for i, row in sub.iterrows():
        ax.annotate(
            f"p = {row['p_holm']:.3g}",
            (row["ci_high"], i),
            xytext=(5, -3),
            textcoords="offset points",
            fontsize=8,
        )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y, sub["compared_with"])
    ax.set(title=label, xlabel="Mean Dice difference, proposed minus comparator (95% CI)")
fig.tight_layout()
save_fig(fig, "06_paired_differences")

## 6. External validation and comparison with published 2D results

### Generalisation to BraTS-Africa

The same models, thresholds and weights, chosen on BraTS 2020 validation cases, applied unchanged to the 60 BraTS-Africa patients (lower-field scanners, lower resolution and later presentation than the BraTS 2020 centres). The gap is external minus internal Dice per region. Expected output: the external table, the gap table and a figure.

In [ ]:
display(RESULTS["external"].round(4))
gap_rows = []
for name in ROW_ORDER:
    if name not in RESULTS["test"].index or name not in RESULTS["external"].index:
        continue
    row = {"model": name}
    for column in ("dice_mean", *[f"dice_{r}" for r in REGIONS]):
        row[f"internal_{column}"] = RESULTS["test"].loc[name, column]
        row[f"external_{column}"] = RESULTS["external"].loc[name, column]
        row[f"gap_{column}"] = row[f"external_{column}"] - row[f"internal_{column}"]
    gap_rows.append(row)
GAP = pd.DataFrame(gap_rows).set_index("model")
save_table(GAP.round(4), "generalisation_gap", index=True)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8), sharey=True)
for ax, column in zip(axes, ("dice_mean", *[f"dice_{r}" for r in REGIONS])):
    x = np.arange(len(GAP))
    ax.bar(x - 0.2, GAP[f"internal_{column}"], 0.4, label="Internal test")
    ax.bar(x + 0.2, GAP[f"external_{column}"], 0.4, label="External")
    ax.set_xticks(x, GAP.index, rotation=35, ha="right", fontsize=8)
    ax.set(title=column.replace("dice_", "Dice ").replace("mean", "(mean of regions)"), ylim=(0, 1))
axes[0].legend()
fig.tight_layout()
save_fig(fig, "07_internal_vs_external")

### The 2D slice-wise protocol

Many 2D studies keep only the axial slices that contain the region, then average the Dice over those slices. That protocol cannot penalise false positives on tumour-free slices and is not comparable with per-patient 3D Dice. Every model here is scored both ways: mean per-patient 3D Dice (the BraTS protocol), the slice-wise mean over tumour-containing slices, and the Dice pooled over those slices. The published GCSER-UNet ensemble (`REFERENCE_2D`) used the slice-wise protocol with a 75:15:10 split of the processed slices; only its own numbers are shown, nothing of it is rerun. Expected output: a table and a figure.

In [ ]:
slice_rows = []
for role, label in ROLES.items():
    for name in ROW_ORDER:
        sub = ALL_CASES[role][ALL_CASES[role]["model"] == name]
        if sub.empty:
            continue
        row = {"set": label, "model": name}
        for region in REGIONS:
            row[f"case_3d_{region}"] = sub[f"dice_{region}"].mean()
            row[f"slice_mean_{region}"] = sub[f"slice_dice_sum_{region}"].sum() / max(
                sub[f"slices_{region}"].sum(), 1
            )
            row[f"slice_pooled_{region}"] = (
                2
                * sub[f"slice_overlap_{region}"].sum()
                / max(sub[f"slice_total_{region}"].sum(), 1)
            )
        slice_rows.append(row)
slice_rows.append(
    {
        "set": "Published (own split)",
        "model": REFERENCE_2D["model"],
        **{f"slice_mean_{r}": REFERENCE_2D[r] for r in REGIONS},
    }
)
SLICE = pd.DataFrame(slice_rows)
save_table(SLICE.round(4), "slice_protocol_comparison")
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
proposed = SLICE[SLICE["model"] == PROPOSED].set_index("set")
for ax, region in zip(axes, REGIONS):
    bars = {"GCSER-UNet (published, slice-wise)": REFERENCE_2D[region]}
    for label in ROLES.values():
        if label in proposed.index:
            bars[f"Proposed, {label}\nslice-wise"] = proposed.loc[label, f"slice_mean_{region}"]
            bars[f"Proposed, {label}\nper-patient 3D"] = proposed.loc[label, f"case_3d_{region}"]
    ax.bar(
        range(len(bars)),
        list(bars.values()),
        color=["grey"] + ["tab:blue", "tab:cyan", "tab:orange", "tab:red"][: len(bars) - 1],
    )
    for i, value in enumerate(bars.values()):
        ax.text(i, value + 0.01, f"{value:.3f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(bars)), list(bars.keys()), rotation=35, ha="right", fontsize=7)
    ax.set(title=f"{region} Dice under each protocol", ylim=(0, 1.05))
fig.tight_layout()
save_fig(fig, "08_protocol_comparison")

### Results by tumour grade

Mean Dice per region for internal HGG and LGG test cases separately, with bootstrap CIs of the mean Dice, to check whether models trained on a mostly HGG cohort underperform on LGG (where ET is often absent). Expected output: a table and a figure.

In [ ]:
grade_rows = []
internal_cases = ALL_CASES["test"]
for (name, grade), sub in internal_cases.groupby(["model", "grade"]):
    low, high = lib.bootstrap_mean_ci(sub["dice_mean"], BOOTSTRAP_SAMPLES, SPLIT_SEED)
    grade_rows.append(
        {
            "model": name,
            "grade": grade,
            "cases": len(sub),
            **{f"dice_{r}": sub[f"dice_{r}"].mean() for r in REGIONS},
            "dice_mean": sub["dice_mean"].mean(),
            "ci_low": low,
            "ci_high": high,
        }
    )
BY_GRADE = pd.DataFrame(grade_rows)
save_table(BY_GRADE.round(4), "test_results_by_grade")
fig, ax = plt.subplots(figsize=(14, 4.5))
grades_present = sorted(BY_GRADE["grade"].unique())
for k, grade in enumerate(grades_present):
    sub = BY_GRADE[BY_GRADE["grade"] == grade].set_index("model").reindex(ROW_ORDER)
    x = np.arange(len(ROW_ORDER)) + (k - (len(grades_present) - 1) / 2) * 0.35
    ax.bar(
        x,
        sub["dice_mean"],
        0.35,
        yerr=[sub["dice_mean"] - sub["ci_low"], sub["ci_high"] - sub["dice_mean"]],
        capsize=3,
        label=f"{grade} ({int(sub['cases'].max())} cases)",
    )
ax.set_xticks(range(len(ROW_ORDER)), ROW_ORDER, rotation=25, ha="right", fontsize=8)
ax.set(
    title="Internal test mean Dice by tumour grade (95% bootstrap CI)",
    ylabel="Mean Dice",
    ylim=(0, 1),
)
ax.legend()
save_fig(fig, "09_dice_by_grade")

### Per-case distributions

Box plots of per-case Dice and HD95 for every region, member and ensemble, internal and external. Expected output: one figure.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(20, 20))
for block, role in enumerate(ROLES):
    for col, region in enumerate(REGIONS):
        for offset, metric in enumerate(("dice", "hd95")):
            ax = axes[2 * block + offset, col]
            data = [
                ALL_CASES[role].loc[ALL_CASES[role]["model"] == m, f"{metric}_{region}"].dropna()
                for m in ROW_ORDER
            ]
            ax.boxplot(data, showfliers=True)
            ax.set_xticks(
                range(1, len(ROW_ORDER) + 1), ROW_ORDER, rotation=35, ha="right", fontsize=7
            )
            ax.set_title(
                f"{ROLES[role]}: {region} {'Dice' if metric == 'dice' else 'HD95 (mm, log scale)'}",
                fontsize=9,
            )
            if metric == "hd95":
                ax.set_yscale("log")
fig.tight_layout()
save_fig(fig, "10_per_case_distributions")

### Qualitative comparison

For the internal and external cases with the median and the lowest proposed-ensemble Dice, the axial slice with the largest reference tumour: the reference, every member and the proposed ensemble on FLAIR. Colours: oedema-only WT in green, TC in yellow, ET in red. Expected output: one figure.

In [ ]:
def overlay(ax, flair, regions, title):
    ax.imshow(np.rot90(flair), cmap="gray")
    colour = np.zeros((*flair.shape, 4))
    colour[regions[0]] = (0.0, 0.8, 0.0, 0.45)
    colour[regions[1]] = (1.0, 0.85, 0.0, 0.55)
    colour[regions[2]] = (1.0, 0.0, 0.0, 0.65)
    ax.imshow(np.rot90(colour, axes=(0, 1)))
    ax.set_title(title, fontsize=8)
    ax.axis("off")


picks = []
for role in ROLES:
    scores = ALL_CASES[role][ALL_CASES[role]["model"] == PROPOSED].set_index("case")["dice_mean"]
    picks.append((role, "median", (scores - scores.median()).abs().idxmin()))
    picks.append((role, "lowest", scores.idxmin()))
columns = FUSION_MEMBERS + [PROPOSED]
fig, axes = plt.subplots(
    len(picks),
    len(columns) + 1,
    figsize=(2.9 * (len(columns) + 1), 3.2 * len(picks)),
    squeeze=False,
)
for row, (role, kind, case) in enumerate(picks):
    reference = lib.load_regions(CASE_CACHE, case)
    z = int(np.argmax(reference[0].sum(axis=(0, 1))))
    flair = np.asarray(lib.load_image(CASE_CACHE, case)[0, :, :, z], dtype=np.float32)
    overlay(
        axes[row, 0],
        flair,
        reference[:, :, :, z],
        f"Reference\n{case} ({ROLES[role].split(' (')[0]}, {kind})",
    )
    for col, name in enumerate(columns, start=1):
        if name in FUSION_MEMBERS:
            pred = lib.regions_from_probs(
                torch.from_numpy(member_probs(name, role, case)), MEMBER_ET_MIN[name]
            ).numpy()
        else:
            pred = (
                lib.regions_from_probs(
                    fused_probs(name, role, case), ET_MIN[name], ENSEMBLES[name][1]
                )
                .cpu()
                .numpy()
            )
        score = ALL_CASES[role][
            (ALL_CASES[role]["model"] == name) & (ALL_CASES[role]["case"] == case)
        ]["dice_mean"].iloc[0]
        overlay(
            axes[row, col], flair, pred[:, :, :, z], f"{name.split(' (')[0]}\nmean Dice {score:.3f}"
        )
fig.tight_layout()
save_fig(fig, "11_qualitative_comparison")

## 7. Uncertainty and quality control

Two per-case warning signals are compared: the TTA spread (mean flip standard deviation inside the predicted WT, averaged over members) and the **architectural disagreement** (1 minus the mean pairwise Dice between the members' masks, averaged over the three regions). A useful signal is high for poor cases: a negative Spearman correlation with the proposed ensemble's per-case mean Dice, and a high AUROC for flagging failures (mean Dice below `QC_FAIL_DICE`). Expected output: a table and a figure.

In [ ]:
qc_rows, qc_cases = [], []
for role, frame in EVAL_CASES.items():
    proposed_scores = ALL_CASES[role][ALL_CASES[role]["model"] == PROPOSED].set_index("case")[
        "dice_mean"
    ]
    spread = MEMBER_CASES[role].groupby("case")["tta_std_in_wt"].mean()
    for case in tqdm(frame["case"], desc=f"Disagreement ({role})"):
        masks = [
            lib.regions_from_probs(
                torch.from_numpy(member_probs(m, role, case)).to(DEVICE), MEMBER_ET_MIN[m]
            )
            for m in FUSION_MEMBERS
        ]
        pairs = [(a, b) for i, a in enumerate(masks) for b in masks[i + 1 :]]
        agreement = np.mean([[lib.dice(a[r], b[r]) for r in range(3)] for a, b in pairs])
        qc_cases.append(
            {
                "set": ROLES[role],
                "case": case,
                "disagreement": 1 - agreement,
                "tta_spread": spread[case],
                "dice_mean": proposed_scores[case],
            }
        )
QC_CASES = pd.DataFrame(qc_cases)
QC_CASES["failure"] = QC_CASES["dice_mean"] < QC_FAIL_DICE
QC_CASES.to_csv(METRICS / "qc_per_case.csv", index=False)
for label, sub in QC_CASES.groupby("set"):
    for signal in ("disagreement", "tta_spread"):
        rho, p = spearmanr(sub[signal], sub["dice_mean"])
        both = sub["failure"].nunique() == 2
        qc_rows.append(
            {
                "set": label,
                "signal": signal,
                "cases": len(sub),
                "failures": int(sub["failure"].sum()),
                "spearman_rho": rho,
                "p_value": p,
                "auroc_failure": roc_auc_score(sub["failure"], sub[signal]) if both else np.nan,
            }
        )
QC = pd.DataFrame(qc_rows)
save_table(QC.round(4), "quality_control")
fig, axes = plt.subplots(2, len(ROLES), figsize=(15, 10), squeeze=False)
for col, label in enumerate(ROLES.values()):
    sub = QC_CASES[QC_CASES["set"] == label]
    for row, signal in enumerate(("disagreement", "tta_spread")):
        ax = axes[row, col]
        ax.scatter(
            sub[signal],
            sub["dice_mean"],
            c=np.where(sub["failure"], "tab:red", "tab:blue"),
            s=18,
            alpha=0.7,
        )
        ax.axhline(QC_FAIL_DICE, color="grey", linestyle="--", linewidth=1)
        stats = QC[(QC["set"] == label) & (QC["signal"] == signal)].iloc[0]
        ax.set(
            title=f"{label}: rho {stats['spearman_rho']:.2f}, AUROC {stats['auroc_failure']:.2f}",
            xlabel=signal.replace("_", " "),
            ylabel="Proposed mean Dice",
        )
fig.tight_layout()
save_fig(fig, "12_quality_control")

## 8. Explainability

### Seg-Grad-CAM maps

For the internal and external cases with the median proposed-ensemble Dice, a 128³ crop centred on the reference tumour goes through every member; Seg-Grad-CAM (gradient-weighted activations of a decoder stage at 1/4 resolution, `RECIPES[...]["cam_layer"]`) is computed for each region, with the target restricted to the voxels the member assigns to that region. Each panel shows FLAIR, the map, and the reference region outline in white. Expected output: one figure per case.

In [ ]:
CAM_MODELS = {}
for name in FUSION_MEMBERS:
    run = SUMMARIES[SUMMARIES["model"] == name].sort_values("best_val_dice").iloc[-1]
    model = lib.build_model(name)
    model.load_state_dict(
        torch.load(
            WEIGHTS / lib.slug(name) / f"{run['run_id']}.pth", map_location="cpu", weights_only=True
        )
    )
    CAM_MODELS[name] = model.to(DEVICE).eval()


def cam_input(case):
    """128^3 crop centred on the reference whole tumour: image (1, 4, P, P, P) and regions (3, P, P, P)."""
    regions = lib.load_regions(CASE_CACHE, case)
    points = np.argwhere(regions[0])
    centre = points.mean(0).round().astype(int) if len(points) else np.array(regions.shape[1:]) // 2
    start = centre - PATCH // 2
    image = lib.crop_pad(np.asarray(lib.load_image(CASE_CACHE, case)), start, PATCH).astype(
        np.float32
    )
    return torch.from_numpy(image)[None].to(DEVICE), lib.crop_pad(regions, start, PATCH)


for role in ROLES:
    scores = ALL_CASES[role][ALL_CASES[role]["model"] == PROPOSED].set_index("case")["dice_mean"]
    case = (scores - scores.median()).abs().idxmin()
    x, reference = cam_input(case)
    z = int(np.argmax(reference[0].sum(axis=(0, 1))))
    flair = x[0, 0, :, :, z].cpu().numpy()
    fig, axes = plt.subplots(
        len(FUSION_MEMBERS), 3, figsize=(12, 4 * len(FUSION_MEMBERS)), squeeze=False
    )
    for row, name in enumerate(FUSION_MEMBERS):
        for col, region in enumerate(REGIONS):
            cam, _ = lib.seg_grad_cam(CAM_MODELS[name], x, lib.RECIPES[name]["cam_layer"], col)
            ax = axes[row, col]
            ax.imshow(np.rot90(flair), cmap="gray")
            ax.imshow(np.rot90(cam[:, :, z].cpu().numpy()), cmap="jet", alpha=0.45, vmin=0, vmax=1)
            if reference[col][:, :, z].any():
                ax.contour(
                    np.rot90(reference[col][:, :, z].astype(float)),
                    levels=[0.5],
                    colors="white",
                    linewidths=1,
                )
            ax.set_title(f"{name}: {region}", fontsize=9)
            ax.axis("off")
    fig.suptitle(f"Seg-Grad-CAM, {ROLES[role]}, {case} (median proposed Dice {scores[case]:.3f})")
    fig.tight_layout()
    save_fig(fig, f"13_gradcam_{role}")

### Quantitative attribution

For every internal test and external case, the share of each member's Seg-Grad-CAM mass that falls inside the reference region dilated by `CAM_MARGIN` voxels, against the share expected by chance (the dilated region's fraction of the crop). Enrichment above 1 means the evidence the model uses concentrates on the tumour region. Regions absent from the reference are skipped. Expected output: a table and a figure.

In [ ]:
cam_rows = []
for role, frame in EVAL_CASES.items():
    for case in tqdm(frame["case"], desc=f"Seg-Grad-CAM attribution ({role})"):
        x, reference = cam_input(case)
        for name, model in CAM_MODELS.items():
            for index, region in enumerate(REGIONS):
                if not reference[index].any():
                    continue
                cam, _ = lib.seg_grad_cam(model, x, lib.RECIPES[name]["cam_layer"], index)
                inside, chance = lib.attribution_in_region(
                    cam, torch.from_numpy(reference[index]).to(DEVICE), CAM_MARGIN
                )
                cam_rows.append(
                    {
                        "set": ROLES[role],
                        "case": case,
                        "model": name,
                        "region": region,
                        "inside": inside,
                        "chance": chance,
                    }
                )
CAM_CASES = pd.DataFrame(cam_rows)
CAM_CASES["enrichment"] = CAM_CASES["inside"] / CAM_CASES["chance"]
CAM_CASES.to_csv(METRICS / "gradcam_attribution_per_case.csv", index=False)
CAM_SUMMARY = (
    CAM_CASES.groupby(["set", "model", "region"])[["inside", "chance", "enrichment"]]
    .mean()
    .reset_index()
)
save_table(CAM_SUMMARY.round(4), "gradcam_attribution")
fig, axes = plt.subplots(1, len(ROLES), figsize=(15, 4.5), sharey=True, squeeze=False)
for ax, label in zip(axes[0], ROLES.values()):
    sub = CAM_SUMMARY[CAM_SUMMARY["set"] == label]
    for i, name in enumerate(FUSION_MEMBERS):
        values = sub[sub["model"] == name].set_index("region").reindex(REGIONS)["inside"]
        ax.bar(np.arange(3) + (i - 1) * 0.27, values, 0.27, color=COLOURS[name], label=name)
    chance = sub.groupby("region")["chance"].mean().reindex(REGIONS)
    ax.scatter(np.arange(3), chance, color="black", marker="_", s=600, label="chance")
    ax.set_xticks(range(3), REGIONS)
    ax.set(title=label, ylabel="Share of attribution inside the region", ylim=(0, 1))
    ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "14_gradcam_attribution")
del CAM_MODELS
torch.cuda.empty_cache()

### Sequence-drop ablation

Each worker re-predicted every internal test case with one MRI sequence at a time set to zero (the mean of its z-scored brain voxels), without TTA. The Dice drop against the full input shows which sequence each member relies on for each region; the expected clinical pattern is FLAIR for WT and T1CE for TC and ET. Expected output: a table and a heatmap per region.

In [ ]:
ablation = pd.concat(
    [
        pd.read_csv(RUNS / s["run_id"] / "modality_ablation.csv").assign(model=s["model"])
        for s in SUMMARIES.to_dict("records")
    ],
    ignore_index=True,
)
mean_dice = ablation.groupby(["model", "dropped"])[[f"dice_{r}" for r in REGIONS]].mean()
drops = []
for name in FUSION_MEMBERS:
    full = mean_dice.loc[(name, "none")]
    for modality in lib.MODALITIES:
        drops.append(
            {
                "model": name,
                "dropped": modality,
                **{
                    f"drop_{r}": full[f"dice_{r}"] - mean_dice.loc[(name, modality), f"dice_{r}"]
                    for r in REGIONS
                },
            }
        )
DROPS = pd.DataFrame(drops)
save_table(DROPS.round(4), "modality_ablation_summary")
fig, axes = plt.subplots(1, 3, figsize=(18, 0.9 * len(FUSION_MEMBERS) + 2.5))
for ax, region in zip(axes, REGIONS):
    grid = DROPS.pivot(index="model", columns="dropped", values=f"drop_{region}").reindex(
        index=FUSION_MEMBERS, columns=list(lib.MODALITIES)
    )
    image = ax.imshow(
        grid.to_numpy(),
        cmap="magma",
        vmin=0,
        vmax=max(0.05, float(np.nanmax(DROPS[[f"drop_{r}" for r in REGIONS]].to_numpy()))),
    )
    ax.set_xticks(range(4), [m.upper() for m in lib.MODALITIES])
    ax.set_yticks(range(len(FUSION_MEMBERS)), FUSION_MEMBERS)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(
                j, i, f"{grid.iat[i, j]:.3f}", ha="center", va="center", color="white", fontsize=8
            )
    ax.grid(False)
    ax.set_title(f"{region}: Dice drop when a sequence is removed")
fig.colorbar(image, ax=axes, shrink=0.8)
save_fig(fig, "15_sequence_drop")

## 9. Efficiency and summary

### Efficiency

Parameters, GFLOPs per 128³ patch, training time and patches seen, peak training and inference memory, and latency per case (sliding window over the median test crop, fp16, without and with the 8-flip TTA). An ensemble costs the sum of its members. Expected output: a table and a figure.

In [ ]:
eff = SUMMARIES.groupby("model").agg(
    seeds=("seed", "count"),
    epochs_run=("epochs_run", "mean"),
    patches_seen=("patches_seen", "mean"),
    micro_batch=("micro_batch", "first"),
    train_hours=("train_seconds", lambda s: s.mean() / 3600),
    peak_train_memory_gb=("peak_train_memory_gb", "max"),
    peak_inference_memory_gb=("peak_inference_memory_gb", "max"),
    latency_s=("latency_s", "mean"),
    latency_tta_s=("latency_tta_s", "mean"),
)
EFFICIENCY = PROFILES[["parameters_m", "gflops_per_patch"]].join(eff, how="inner")
member_rows = EFFICIENCY.loc[FUSION_MEMBERS]
for name, (weights, _) in ENSEMBLES.items():
    used = [m for i, m in enumerate(FUSION_MEMBERS) if weights[:, i].max() > 0]
    EFFICIENCY.loc[
        name, ["parameters_m", "gflops_per_patch", "train_hours", "latency_s", "latency_tta_s"]
    ] = (
        member_rows.loc[
            used, ["parameters_m", "gflops_per_patch", "train_hours", "latency_s", "latency_tta_s"]
        ]
        .sum()
        .to_numpy()
    )
EFFICIENCY = EFFICIENCY.join(
    RESULTS["test"][["dice_mean"]].rename(columns={"dice_mean": "internal_dice_mean"})
)
EFFICIENCY = EFFICIENCY.join(
    RESULTS["external"][["dice_mean"]].rename(columns={"dice_mean": "external_dice_mean"})
)
save_table(EFFICIENCY.round(4), "efficiency", index=True)
save_table(
    SUMMARIES[["run_id", "et_min_voxels", "best_epoch", "best_val_dice"]].round(4),
    "et_threshold_and_selection",
)
fig, ax = plt.subplots(figsize=(10, 6))
for name, row in EFFICIENCY.iterrows():
    ax.scatter(
        row["latency_tta_s"],
        row["internal_dice_mean"],
        s=30 + 3 * row["parameters_m"],
        alpha=0.6,
        color=COLOURS.get(name),
    )
    ax.annotate(
        f"{name}\n{row['parameters_m']:.1f}M",
        (row["latency_tta_s"], row["internal_dice_mean"]),
        fontsize=8,
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set(
    title="Internal test mean Dice vs latency per case with TTA (bubble area: parameters)",
    xlabel="Latency (s per case, 8-flip TTA)",
    ylabel="Mean Dice",
)
save_fig(fig, "16_dice_vs_latency")

### Summary

The headline table (internal and external Dice and HD95 for every member and ensemble), the run environment, the runs still pending, and the limitations that remain. Expected output: the summary table and `metrics/run_summary.json`.

In [ ]:
columns = [
    "kind",
    "dice_mean",
    "dice_mean_ci",
    *[f"dice_{r}" for r in REGIONS],
    *[f"hd95_{r}" for r in REGIONS],
]
summary_table = (
    RESULTS["test"][columns]
    .add_prefix("internal_")
    .join(RESULTS["external"][columns[1:]].add_prefix("external_"))
)
summary_table = summary_table.join(EFFICIENCY[["parameters_m", "latency_tta_s"]])
save_table(summary_table.round(4), "summary_table", index=True)
proposed_slice = SLICE[SLICE["model"] == PROPOSED].set_index("set")
run_summary = {
    "cases": CASES["role"].value_counts().to_dict(),
    "members": FUSION_MEMBERS,
    "epochs": EPOCHS,
    "iters_per_epoch": ITERS_PER_EPOCH,
    "seeds": SEEDS,
    "tta": TTA,
    "deterministic": DETERMINISTIC,
    "finished_runs": SUMMARIES["run_id"].tolist(),
    "pending_runs": pending,
    "fusion": {
        region: {
            "weights": dict(zip(FUSION_MEMBERS, map(float, best_weights[r]))),
            "threshold": best_thresholds[r],
        }
        for r, region in enumerate(REGIONS)
    },
    "et_min_voxels": ET_MIN,
    "internal_dice_mean": RESULTS["test"]["dice_mean"].round(4).to_dict(),
    "external_dice_mean": RESULTS["external"]["dice_mean"].round(4).to_dict(),
    "proposed_slice_protocol": {
        label: {r: float(proposed_slice.loc[label, f"slice_mean_{r}"]) for r in REGIONS}
        for label in proposed_slice.index
    },
    "reference_2d": REFERENCE_2D,
    "quality_control": QC.to_dict("records"),
    "training_wall_minutes": TRAINING_WALL_SECONDS / 60,
    "notebook_minutes": (time.time() - NOTEBOOK_START) / 60,
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "monai": monai.__version__,
        "cuda": torch.version.cuda,
        "gpus": [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
    },
    "limitations": [
        f"{len(SEEDS)} seed(s) per member; seed-to-seed variance of the ensemble is not measured with one seed",
        "fusion weights, thresholds and the ET size filter are selected on 56 validation cases; the test and external sets are never used for selection",
        "BraTS-Africa shares the BraTS annotation protocol and preprocessing (co-registration, skull stripping), so it tests acquisition and population shift, not protocol shift",
        "the comparison with 2D studies uses their published numbers on their own splits",
        "the web app still runs the earlier 128^3 VNet; this notebook does not export a deployment checkpoint",
    ],
}
(METRICS / "run_summary.json").write_text(json.dumps(run_summary, indent=2, default=float))
log(
    f"Proposed ensemble: internal mean Dice {RESULTS['test'].loc[PROPOSED, 'dice_mean']:.4f}, external {RESULTS['external'].loc[PROPOSED, 'dice_mean']:.4f}; pending runs: {len(pending)}"
)

### Package the outputs

Zips everything in `/kaggle/working/` (the volume cache lives in `/tmp` and is not included), storing already-compressed files without recompression, and shows a download link. The zip also appears in the notebook's **Output** tab after a saved run. Expected output: the file count and a download link.

In [ ]:
ZIP_PATH = WORK / "outputs.zip"
EXCLUDE_PARTS = {"__pycache__", ".cache", ".ipynb_checkpoints", "cache", "tmp", "wandb"}
STORED_SUFFIXES = {
    ".zip",
    ".png",
    ".jpg",
    ".jpeg",
    ".pt",
    ".pth",
    ".bin",
    ".safetensors",
    ".gz",
    ".npz",
    ".npy",
}
files = [
    path
    for path in WORK.rglob("*")
    if path.is_file()
    and path != ZIP_PATH
    and not EXCLUDE_PARTS.intersection(path.relative_to(WORK).parts)
]
total_bytes = sum(path.stat().st_size for path in files)
free_bytes = shutil.disk_usage(WORK).free
assert (
    free_bytes > total_bytes * 1.05
), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
    for path in tqdm(files, desc="Zipping outputs", unit="file"):
        method = (
            zipfile.ZIP_STORED if path.suffix.lower() in STORED_SUFFIXES else zipfile.ZIP_DEFLATED
        )
        archive.write(path, path.relative_to(WORK), compress_type=method)
print(f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}", flush=True)
display(FileLink(os.path.relpath(ZIP_PATH)))